In [15]:
### 업로드할 파일: 총 4개





### 셀 1 — 파일 업로드 및 데이터 확인


# ============================================================
# 셀 1. 새 실험 시작: 시드 고정 → 원본 CSV 업로드 → 데이터 확인
# Google Colab용. 이전 셀이나 저장된 모델에 의존하지 않습니다.
# ============================================================
import os
import random
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import files

# 1. 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.set_num_threads(1)

# 2. 매번 새 폴더를 사용하여 이전 실행 파일과 섞이지 않도록 함
RUN_DIR = Path(tempfile.mkdtemp(prefix="taxi_run_", dir="/content"))
DATA_DIR = RUN_DIR / "data"
OUTPUT_DIR = RUN_DIR / "output"
DATA_DIR.mkdir()
OUTPUT_DIR.mkdir()

required_files = (
    "nodes.csv",
    "edges.csv",
    "train_requests.csv",
    "test_requests.csv",
)

print("다음 원본 CSV 4개를 한꺼번에 선택하세요:")
for name in required_files:
    print(" -", name)

# 빈 폴더에서 업로드하므로 Colab이 중복 파일명에 '(1)'을 붙이는 것을 방지
previous_dir = Path.cwd()
try:
    os.chdir(DATA_DIR)
    uploaded = files.upload()
finally:
    os.chdir(previous_dir)

missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"누락된 파일: {missing}\n"
        f"실제 업로드 파일: {list(uploaded)}\n"
        "파일명이 표의 이름과 정확히 일치하는지 확인한 뒤 셀을 다시 실행하세요."
    )

for name in required_files:
    if (DATA_DIR / name).stat().st_size == 0:
        raise ValueError(f"빈 파일입니다: {name}")
del uploaded

# 3. 도로망과 학습 데이터의 형식 확인
# 테스트 요청 내용은 지금 읽지 않고 최종 평가 단계까지 보관
nodes = pd.read_csv(DATA_DIR / "nodes.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")
train_requests = pd.read_csv(DATA_DIR / "train_requests.csv")

expected_columns = {
    "nodes.csv": (nodes, {"node_id"}),
    "edges.csv": (
        edges,
        {"from", "to", "base_min", "distance_km", "congestion_sensitivity"},
    ),
    "train_requests.csv": (
        train_requests,
        {"day", "request_id", "minute", "pickup", "dropoff", "max_wait", "base_fare"},
    ),
}

for name, (df, required_cols) in expected_columns.items():
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"{name}에 필요한 컬럼이 없습니다: {sorted(missing_cols)}")
    if df[list(required_cols)].isna().any().any():
        raise ValueError(f"{name}의 필수 컬럼에 결측값이 있습니다.")

if not nodes["node_id"].is_unique:
    raise ValueError("nodes.csv의 node_id가 중복되었습니다.")
if not train_requests["request_id"].is_unique:
    raise ValueError("train_requests.csv의 request_id가 중복되었습니다.")
if set(train_requests["day"].unique()) != {1, 2, 3}:
    raise ValueError("학습 요청의 날짜가 Day 1~3인지 확인하세요.")

node_ids = set(nodes["node_id"])
for df, col in [
    (edges, "from"), (edges, "to"),
    (train_requests, "pickup"), (train_requests, "dropoff"),
]:
    unknown = set(df[col]) - node_ids
    if unknown:
        raise ValueError(f"{col}에 nodes.csv에 없는 노드가 있습니다: {unknown}")

# 4. 확인 결과
print("\n===== 셀 1 완료 =====")
print("데이터 폴더:", DATA_DIR)
print("결과 저장 폴더:", OUTPUT_DIR)
print(f"노드: {len(nodes)}개 | 도로: {len(edges)}개")
print("\n[학습일별 승객 요청 수]")
print(train_requests.groupby("day").size().rename("요청 수").to_string())
print("\ntest_requests.csv: 업로드 완료 (내용은 아직 읽지 않음)")





다음 원본 CSV 4개를 한꺼번에 선택하세요:
 - nodes.csv
 - edges.csv
 - train_requests.csv
 - test_requests.csv


Saving edges.csv to edges.csv
Saving nodes.csv to nodes.csv
Saving test_requests.csv to test_requests.csv
Saving train_requests.csv to train_requests.csv

===== 셀 1 완료 =====
데이터 폴더: /content/taxi_run_n4mv6549/data
결과 저장 폴더: /content/taxi_run_n4mv6549/output
노드: 30개 | 도로: 60개

[학습일별 승객 요청 수]
day
1    400
2    400
3    400

test_requests.csv: 업로드 완료 (내용은 아직 읽지 않음)


In [16]:
# ============================================================
# 셀 2. 대회 공식 환경 구성 — 환경 로직 수정 금지
# ============================================================
import networkx as nx

MAX_CANDIDATES, START, END = 8, 7 * 60, 22 * 60
FUEL_PER_KM, CENTER_STRENGTH, START_NODE = 150, 2.5, 0


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)


def load_data(data_dir=Path("."), split="train"):
    nodes = pd.read_csv(data_dir / "nodes.csv")
    edges = pd.read_csv(data_dir / "edges.csv")
    requests = pd.read_csv(data_dir / f"{split}_requests.csv")

    attrs = ["base_min", "distance_km", "congestion_sensitivity"]
    required = {"from", "to", *attrs}
    if not required.issubset(edges.columns):
        raise ValueError(f"Missing edge columns: {required - set(edges.columns)}")

    g = nx.from_pandas_edgelist(edges, "from", "to", edge_attr=attrs)
    g.add_nodes_from(nodes["node_id"].astype(int))

    if START_NODE not in g:
        raise ValueError(f"Start node {START_NODE} is missing")
    if requests["request_id"].duplicated().any():
        raise ValueError("Duplicate request_id")
    if not nx.is_connected(g):
        raise ValueError("Road graph must be connected")
    if not edges["congestion_sensitivity"].between(0, 1).all():
        raise ValueError("Invalid congestion_sensitivity")

    print(
        f"Loaded {g.number_of_nodes()} nodes, {g.number_of_edges()} edges, "
        f"{len(requests)} {split} requests"
    )
    return g, requests


class TaxiEnv:
    """행동 0=대기, 1~8=후보 승객 선택. 공식 점수=요금+팁-유류비."""

    TRAFFIC = [
        0.8, 0.8, 0.8, 0.8, 0.9, 1.0, 1.2, 1.5,
        1.7, 1.4, 1.1, 1.1, 1.3, 1.3, 1.2, 1.2,
        1.3, 1.5, 1.8, 1.6, 1.3, 1.1, 1.0, 0.9
    ]
    CENTER_STRENGTH = 2.5
    TIPS = {1: 0, 2: 0, 3: 0.05, 4: 0.1, 5: 0.2}

    def __init__(self, graph, requests, day):
        self.g, self.all_requests, self.day = graph, requests, day
        self.n_actions = MAX_CANDIDATES + 1
        self.hops = dict(nx.all_pairs_shortest_path_length(graph))
        self.reset()

    def reset(self):
        self.time, self.pos = START, START_NODE
        self.earned = self.served = 0
        self.total_fare = self.total_tip = self.total_fuel = 0
        self.completed = set()

        data = self.all_requests[self.all_requests.day == self.day]
        self.reqs = data.sort_values(["minute", "request_id"]).to_dict("records")
        return self.observe()

    def _edge_time(self, edge, minute):
        base = self.TRAFFIC[(int(minute) // 60) % 24]
        factor = max(
            0.5,
            1 + (base - 1) * (
                0.35 + self.CENTER_STRENGTH * edge["congestion_sensitivity"]
            )
        )
        return max(1, int(np.ceil(edge["base_min"] * factor)))

    def shortest_route(self, src, dst, depart):
        if src == dst:
            return [src], 0

        hour = (int(depart) // 60) % 24
        cache = self.g.graph.setdefault("route_cache", {})
        key = (src, dst, hour)

        if key not in cache:
            weight = lambda u, v, d: self._edge_time(d, depart)
            path = nx.shortest_path(
                self.g, src, dst, weight=weight, method="dijkstra"
            )
            time = sum(
                self._edge_time(self.g[u][v], depart)
                for u, v in zip(path, path[1:])
            )
            cache[key] = (path, time)

        return cache[key]

    def active(self):
        return [
            r for r in self.reqs
            if r["minute"] <= self.time < r["minute"] + r["max_wait"]
            and r["request_id"] not in self.completed
        ]

    def candidates(self):
        scored = []

        for r in self.active():
            if self.hops[self.pos][r["pickup"]] > 3:
                continue

            _, p = self.shortest_route(self.pos, r["pickup"], self.time)
            pickup_at = self.time + p
            if pickup_at >= r["minute"] + r["max_wait"]:
                continue

            _, d = self.shortest_route(r["pickup"], r["dropoff"], pickup_at)
            if pickup_at + d > END:
                continue

            score = r["base_fare"] / max(1, p + d)
            scored.append((score, -r["request_id"], r, p, d))

        scored.sort(reverse=True)
        return [(r, p, d) for _, _, r, p, d in scored[:MAX_CANDIDATES]]

    def observe(self):
        choices = self.candidates()
        state = [
            self.pos / 29,
            (self.time - START) / (END - START),
            len(self.active()) / 20,
            self.TRAFFIC[(self.time // 60) % 24] / 2,
        ]
        mask = np.zeros(self.n_actions, dtype=bool)
        mask[0] = True

        for i in range(MAX_CANDIDATES):
            if i < len(choices):
                r, p, d = choices[i]
                mask[i + 1] = True
                state.extend([
                    r["pickup"] / 29,
                    r["dropoff"] / 29,
                    p / 30,
                    d / 45,
                    r["base_fare"] / 12000,
                    (r["minute"] + r["max_wait"] - self.time - p) / 12,
                ])
            else:
                state.extend([0.0] * 6)

        return np.asarray(state, dtype=np.float32), mask

    def step(self, action):
        choices = self.candidates()
        reward, info = 0.0, {}

        if 1 <= action <= len(choices):
            r, p, d = choices[action - 1]

            fraction = min(
                1, (self.time + p - r["minute"]) / r["max_wait"]
            )
            rating = max(1, 5 - min(4, int(fraction * 5)))

            fare = int(r["base_fare"])
            tip_rate = self.TIPS[rating]
            tip = round(fare * tip_rate)

            path, _ = self.shortest_route(self.pos, r["pickup"], self.time)
            empty_km = sum(
                self.g[u][v]["distance_km"]
                for u, v in zip(path, path[1:])
            )
            fuel = round(empty_km * FUEL_PER_KM)
            gross, net = fare + tip, fare + tip - fuel

            self.time += p + d
            self.pos = r["dropoff"]
            self.served += 1
            self.completed.add(r["request_id"])

            self.total_fare += fare
            self.total_tip += tip
            self.total_fuel += fuel
            self.earned += net

            reward = net / 1000
            info = {
                "request_id": r["request_id"],
                "rating": rating,
                "tip_rate": tip_rate,
                "tip": tip,
                "fare": fare,
                "gross": gross,
                "fuel": fuel,
                "net": net,
            }

        else:
            outside = [
                r for r in self.active()
                if self.hops[self.pos][r["pickup"]] > 3
            ]

            if not choices and outside:
                target = min(
                    outside,
                    key=lambda r: (
                        self.hops[self.pos][r["pickup"]],
                        r["request_id"],
                    )
                )
                path, _ = self.shortest_route(
                    self.pos, target["pickup"], self.time
                )
                nxt = path[1]
                delta = self._edge_time(self.g[self.pos][nxt], self.time)
                fuel = round(self.g[self.pos][nxt]["distance_km"] * FUEL_PER_KM)

                self.total_fuel += fuel
                self.earned -= fuel
                self.pos, self.time = nxt, self.time + delta

                reward, info = -fuel / 1000, {"fuel": fuel}
            else:
                self.time += 1
                reward = -0.02  # 학습 보상에만 적용. 공식 순수익 차감 아님.

        done = self.time >= END
        return self.observe(), reward, done, info


# ============================================================
# 작동 확인 — 학습 데이터의 Day 1 초기 상태만 확인
# ============================================================
seed_all(SEED)
graph, train_requests = load_data(DATA_DIR, split="train")

check_env = TaxiEnv(graph, train_requests, day=1)
state, mask = check_env.observe()

assert check_env.time == START and check_env.pos == START_NODE
assert state.shape == (52,)
assert mask.shape == (9,) and mask[0]
assert int(mask.sum()) == 1 + len(check_env.candidates())

print("\n===== 셀 2 완료 =====")
print(f"Day 1 시작: {check_env.time // 60:02d}:{check_env.time % 60:02d}")
print("택시 위치:", check_env.pos)
print("상태 차원:", state.shape)
print("현재 가능한 행동:", np.flatnonzero(mask).tolist())

Loaded 30 nodes, 60 edges, 1200 train requests

===== 셀 2 완료 =====
Day 1 시작: 07:00
택시 위치: 0
상태 차원: (52,)
현재 가능한 행동: [0]


In [17]:
# ============================================================
# 셀 3. 기준 정책 실행: 후보가 있으면 1번, 없으면 대기
# 전제: 셀 1, 셀 2 실행 완료
# ============================================================
MAX_STEPS = 2000
FIT_DAYS = (1, 2)
VALID_DAY = 3


def first_candidate_policy(env, choices):
    return 1 if choices else 0


def run_policy_day(graph, requests, day, policy_fn):
    """공식 환경에서 하루 운행. 정책 비교를 위해 요약과 행동 기록을 반환."""
    env = TaxiEnv(graph, requests, day)
    history = []

    for step in range(MAX_STEPS):
        choices = env.candidates()
        action = policy_fn(env, choices)

        if not isinstance(action, (int, np.integer)) or not 0 <= action <= len(choices):
            raise ValueError(f"Day {day}, step {step}: 유효하지 않은 행동 {action}")

        # step 이후에는 시간·위치·후보가 달라지므로 먼저 기록
        t0, pos0, net0 = env.time, env.pos, env.earned
        pickup_min, trip_min, pid = 0, 0, -1
        if action:
            request, pickup_min, trip_min = choices[action - 1]
            pid = int(request["request_id"])

        _, _, done, info = env.step(action)
        elapsed = env.time - t0

        if action:
            kind = "승객운행"
            assert int(info["request_id"]) == pid
            assert elapsed == pickup_min + trip_min
        elif "fuel" in info:
            kind = "자동이동"
        elif choices:
            kind = "후보거절대기"
        else:
            kind = "후보없는대기"

        history.append({
            "day": day,
            "step": step,
            "start_minute": t0,
            "start_node": pos0,
            "candidate_count": len(choices),
            "action": int(action),
            "passenger_id": pid,
            "kind": kind,
            "elapsed_min": elapsed,
            "pickup_min": pickup_min,
            "trip_min": trip_min,
            # step의 학습 보상이 아니라 실제 순수익 변화량
            "net_change": env.earned - net0,
        })

        if done:
            break
    else:
        raise RuntimeError(f"Day {day}: {MAX_STEPS}회 안에 종료되지 않았습니다.")

    log = pd.DataFrame(history)
    auto_min = int(log.loc[log["kind"] == "자동이동", "elapsed_min"].sum())
    idle_min = int(log.loc[
        log["kind"].isin(["후보거절대기", "후보없는대기"]), "elapsed_min"
    ].sum())
    pickup_total = int(log["pickup_min"].sum())
    trip_total = int(log["trip_min"].sum())

    # 공식 정산 및 시간 집계 검증
    assert int(log["net_change"].sum()) == env.earned
    assert env.earned == env.total_fare + env.total_tip - env.total_fuel
    assert pickup_total + trip_total + auto_min + idle_min == env.time - START

    summary = {
        "day": day,
        "순수익": int(env.earned),
        "승객수": int(env.served),
        "기본요금": int(env.total_fare),
        "팁": int(env.total_tip),
        "유류비": int(env.total_fuel),
        "후보0명_횟수": int((log["candidate_count"] == 0).sum()),
        "후보1명_횟수": int((log["candidate_count"] == 1).sum()),
        "후보2명이상_횟수": int((log["candidate_count"] >= 2).sum()),
        "제자리대기분": idle_min,
        "자동이동분": auto_min,
        "픽업이동분": pickup_total,
        "승객운행분": trip_total,
    }
    return summary, log


# Day 1·2만 실행
summaries, logs = [], []
for day in FIT_DAYS:
    summary, log = run_policy_day(
        graph, train_requests, day, first_candidate_policy
    )
    summaries.append(summary)
    logs.append(log)

baseline_summary = pd.DataFrame(summaries)
baseline_log = pd.concat(logs, ignore_index=True)

print("[1] 공식 순수익")
print(baseline_summary[
    ["day", "순수익", "승객수", "기본요금", "팁", "유류비"]
].to_string(index=False))

print("\n[2] 선택 기회와 시간 사용")
print(baseline_summary[
    ["day", "후보0명_횟수", "후보1명_횟수", "후보2명이상_횟수",
     "제자리대기분", "자동이동분", "픽업이동분", "승객운행분"]
].to_string(index=False))

print(f"\nDay 1·2 합산 기준 순수익: {baseline_summary['순수익'].sum():,}원")

baseline_summary.to_csv(OUTPUT_DIR / "baseline_summary.csv", index=False)
baseline_log.to_csv(OUTPUT_DIR / "baseline_actions.csv", index=False)
print("진단 결과 저장:", OUTPUT_DIR)

[1] 공식 순수익
 day    순수익  승객수   기본요금     팁   유류비
   1 332727   65 335100 20370 22743
   2 323031   63 332100 13605 22674

[2] 선택 기회와 시간 사용
 day  후보0명_횟수  후보1명_횟수  후보2명이상_횟수  제자리대기분  자동이동분  픽업이동분  승객운행분
   1      186       34         31     154     83    192    471
   2      174       34         29     148     63    209    480

Day 1·2 합산 기준 순수익: 655,758원
진단 결과 저장: /content/taxi_run_n4mv6549/output


In [19]:
 ### 셀 4 — 승객 선택 기준을 `기본요금/분`에서 `순수익/분`으로 변경해 비교


# ============================================================
# 셀 4. 승객 순위 실험: 기본요금/분 vs 공식 순수익/분
# Day 1·2만 사용. 대기 규칙은 변경하지 않음.
# ============================================================

def estimate_candidate_net(env, request, pickup_min):
    """현재 관측 가능한 후보의 순수익을 공식 환경과 같은 방식으로 계산."""
    fare = int(request["base_fare"])

    # 1. 팁: 요청 발생부터 실제 픽업까지 승객이 기다린 시간 기준
    pickup_at = env.time + pickup_min
    wait_fraction = min(
        1.0,
        (pickup_at - request["minute"]) / request["max_wait"]
    )
    rating = max(1, 5 - min(4, int(wait_fraction * 5)))
    tip = round(fare * env.TIPS[rating])

    # 2. 빈차 유류비: 픽업 경로의 실제 거리(km) 기준
    path, _ = env.shortest_route(env.pos, request["pickup"], env.time)
    empty_km = sum(
        env.g[a][b]["distance_km"]
        for a, b in zip(path, path[1:])
    )
    fuel = round(empty_km * FUEL_PER_KM)

    return fare + tip - fuel


def net_per_minute_policy(env, choices):
    if not choices:
        return 0

    def score(choice):
        request, pickup_min, trip_min = choice
        net = estimate_candidate_net(env, request, pickup_min)
        return net / max(1, pickup_min + trip_min)

    # 점수가 같으면 공식 후보 순서에서 앞선 승객 선택
    best_index = max(range(len(choices)), key=lambda i: score(choices[i]))
    return best_index + 1


# ------------------------------------------------------------
# Day 1·2 실행 + 공식 정산 일치 검증
# ------------------------------------------------------------
net_summaries = []
net_logs = []

for day in FIT_DAYS:
    predicted_nets = []

    def audited_policy(env, choices):
        action = net_per_minute_policy(env, choices)
        if action:
            request, pickup_min, _ = choices[action - 1]
            predicted_nets.append(estimate_candidate_net(env, request, pickup_min))
        return action

    summary, log = run_policy_day(graph, train_requests, day, audited_policy)

    actual_nets = log.loc[log["action"] > 0, "net_change"].to_numpy()
    assert np.array_equal(np.asarray(predicted_nets), actual_nets), (
        f"Day {day}: 예상 순수익과 공식 정산 금액이 다릅니다."
    )

    net_summaries.append(summary)
    net_logs.append(log)

netrate_summary = pd.DataFrame(net_summaries)
netrate_log = pd.concat(net_logs, ignore_index=True)


# ------------------------------------------------------------
# 같은 날짜끼리 비교
# ------------------------------------------------------------
base = baseline_summary.set_index("day")
new = netrate_summary.set_index("day")

comparison = pd.DataFrame({
    "기준_순수익": base["순수익"],
    "순수익분_정책": new["순수익"],
    "순수익차이": new["순수익"] - base["순수익"],
    "승객수차이": new["승객수"] - base["승객수"],
    "기본요금차이": new["기본요금"] - base["기본요금"],
    "팁차이": new["팁"] - base["팁"],
    "유류비차이": new["유류비"] - base["유류비"],
})

# 새 정책 자신의 운행 경로에서 공식 후보 1번 대신 다른 후보를 고른 횟수
comparison["1번외선택횟수"] = (
    netrate_log.assign(nonfirst=netrate_log["action"] > 1)
    .groupby("day")["nonfirst"].sum()
)

comparison.loc["합계"] = comparison.sum()

print("[Day 1·2 정책 비교 | 차이 = 새 정책 - 기준 정책]")
print(comparison.to_string())

# 동점이면 계산이 단순한 기존 정책 유지
gain = int(comparison.loc["합계", "순수익차이"])
if gain > 0:
    selected_rank_name = "순수익/분"
    selected_rank_policy = net_per_minute_policy
else:
    selected_rank_name = "기본요금/분"
    selected_rank_policy = first_candidate_policy

print(f"\nDay 1·2 합산 순수익 차이: {gain:+,}원")
print("다음 실험의 기준 승객 선택 방식:", selected_rank_name)

comparison.to_csv(OUTPUT_DIR / "rank_policy_comparison.csv")
netrate_log.to_csv(OUTPUT_DIR / "netrate_actions.csv", index=False)

[Day 1·2 정책 비교 | 차이 = 새 정책 - 기준 정책]
     기준_순수익  순수익분_정책  순수익차이  승객수차이  기본요금차이  팁차이  유류비차이  1번외선택횟수
day                                                            
1    332727   331789   -938     -4   -2700 -180  -1942        3
2    323031   318744  -4287     -2   -4500  510    297        1
합계   655758   650533  -5225     -6   -7200  330  -1645        4

Day 1·2 합산 순수익 차이: -5,225원
다음 실험의 기준 승객 선택 방식: 기본요금/분


In [20]:
### 셀 5 — 승객 순위는 유지하고, 수락·대기 결정만 학습


# ============================================================
# 셀 5. Day 1·2에서 수락/대기 정책 학습
# 승객 순위: 셀 4에서 선택된 방식으로 고정
# ============================================================
from itertools import product


def make_accept_policy(rank_policy, threshold, max_waits):
    """
    하루마다 새로 생성하는 정책.
    threshold: 후보의 예상 순수익/분이 이 값 미만이면 대기 고려
    max_waits: 후보가 있는데도 연속으로 기다릴 수 있는 최대 횟수
    """
    wait_streak = 0

    def policy(env, choices):
        nonlocal wait_streak

        if not choices:
            wait_streak = 0
            return 0

        # 누구를 태울지는 기존 순위 정책이 결정
        action = rank_policy(env, choices)

        # threshold=0, max_waits=0이면 기존 '항상 수락' 정책과 동일
        if max_waits == 0 or wait_streak >= max_waits:
            wait_streak = 0
            return action

        request, pickup_min, trip_min = choices[action - 1]
        net = estimate_candidate_net(env, request, pickup_min)
        net_rate = net / max(1, pickup_min + trip_min)

        # 마감 직전에는 다음 기회를 기다릴 시간이 적으므로 기준을 낮춤
        remaining = max(0, END - env.time)
        effective_threshold = threshold * min(1.0, remaining / 30.0)

        if net_rate < effective_threshold:
            wait_streak += 1
            return 0

        wait_streak = 0
        return action

    return policy


# 후보가 있으면 항상 수락하는 설정을 반드시 포함
configs = [(0, 0)] + list(product((250, 350, 450, 550), (1, 2)))

# 수락/대기 효과만 비교하기 위해 승객 순위가 같은 기준 정책을 사용
reference_summary = (
    baseline_summary if selected_rank_name == "기본요금/분" else netrate_summary
)
reference_net = reference_summary.set_index("day")["순수익"].to_dict()

rows = []
for threshold, max_waits in configs:
    row = {
        "수락기준_원분": threshold,
        "최대연속대기": max_waits,
    }

    for day in FIT_DAYS:
        # 날짜가 바뀌면 연속 대기 횟수도 반드시 초기화
        policy = make_accept_policy(selected_rank_policy, threshold, max_waits)
        summary, log = run_policy_day(graph, train_requests, day, policy)

        row[f"D{day}_순수익"] = summary["순수익"]
        row[f"D{day}_차이"] = summary["순수익"] - reference_net[day]
        row[f"D{day}_의도적대기"] = int((log["kind"] == "후보거절대기").sum())

    row["합산차이"] = sum(row[f"D{day}_차이"] for day in FIT_DAYS)
    rows.append(row)

wait_search = pd.DataFrame(rows)

# 대기를 끈 설정은 셀 4에서 선택한 정책과 결과가 정확히 같아야 함
control = wait_search.iloc[0]
for day in FIT_DAYS:
    assert control[f"D{day}_차이"] == 0
    assert control[f"D{day}_의도적대기"] == 0

print(f"[고정한 승객 순위: {selected_rank_name}]")
print("[차이 = 수락·대기 정책 순수익 − 항상 수락 정책 순수익]")
print(
    wait_search.sort_values("합산차이", ascending=False)
    .to_string(index=False)
)

# 학습일이 이틀뿐이므로 보수적으로 선택:
# 한 날짜에서 번 수익으로 다른 날짜의 손해를 덮은 설정은 채택하지 않음.
eligible = wait_search[
    (wait_search["D1_차이"] >= 0)
    & (wait_search["D2_차이"] >= 0)
    & (wait_search["합산차이"] > 0)
]

if eligible.empty:
    best_wait_config = {"threshold": 0, "max_waits": 0}
    print("\n결론: 채택 조건을 만족한 대기 정책이 없어 '항상 수락'을 유지합니다.")
else:
    winner = eligible.sort_values(
        ["합산차이", "최대연속대기", "수락기준_원분"],
        ascending=[False, True, True],
    ).iloc[0]

    best_wait_config = {
        "threshold": int(winner["수락기준_원분"]),
        "max_waits": int(winner["최대연속대기"]),
    }
    print(f"\n학습일 합산 개선: {int(winner['합산차이']):+,}원")

print("다음 검증에 사용할 설정:", best_wait_config)
wait_search.to_csv(OUTPUT_DIR / "wait_policy_search.csv", index=False)

[고정한 승객 순위: 기본요금/분]
[차이 = 수락·대기 정책 순수익 − 항상 수락 정책 순수익]
 수락기준_원분  최대연속대기  D1_순수익  D1_차이  D1_의도적대기  D2_순수익  D2_차이  D2_의도적대기   합산차이
       0       0  332727      0         0  323031      0         0      0
     250       1  332727      0         0  323031      0         0      0
     250       2  332727      0         0  323031      0         0      0
     350       1  325766  -6961         4  328567   5536        13  -1425
     450       1  296701 -36026        29  334311  11280        23 -24746
     550       1  311736 -20991        40  314571  -8460        38 -29451
     350       2  304454 -28273        13  308493 -14538        20 -42811
     450       2  280194 -52533        53  310764 -12267        56 -64800
     550       2  268575 -64152        67  311797 -11234        79 -75386

결론: 채택 조건을 만족한 대기 정책이 없어 '항상 수락'을 유지합니다.
다음 검증에 사용할 설정: {'threshold': 0, 'max_waits': 0}


In [22]:
### 셀 6 — 확정된 정책으로 Day 3 검증 및 저장


# ============================================================
# 셀 6. 정책 확정 → Day 3 검증 → 설정 및 결과 저장
# ============================================================
import json

# Day 1·2에서 선택한 값을 그대로 고정
final_config = {
    "rank_policy": selected_rank_name,
    "threshold": int(best_wait_config["threshold"]),
    "max_waits": int(best_wait_config["max_waits"]),
    "seed": int(SEED),
}

def make_final_policy(config):
    """날짜가 바뀔 때마다 새 정책을 생성해 내부 대기 상태를 초기화."""
    rank_functions = {
        "기본요금/분": first_candidate_policy,
        "순수익/분": net_per_minute_policy,
    }
    return make_accept_policy(
        rank_policy=rank_functions[config["rank_policy"]],
        threshold=config["threshold"],
        max_waits=config["max_waits"],
    )

# Day 3은 파라미터 탐색 없이 딱 한 번 운행
valid_summary, valid_log = run_policy_day(
    graph,
    train_requests,
    VALID_DAY,
    make_final_policy(final_config),
)

# 이번에 선택한 설정이 실제 의도대로 행동했는지 확인
if final_config["rank_policy"] == "기본요금/분" and final_config["max_waits"] == 0:
    expected_actions = (valid_log["candidate_count"] > 0).astype(int)
    assert np.array_equal(valid_log["action"].to_numpy(), expected_actions.to_numpy())
    print("행동 검증 통과: 후보가 없으면 0번, 있으면 항상 1번을 선택했습니다.")

valid_df = pd.DataFrame([valid_summary])
print("\n[Day 3 공식 검증 결과]")
print(valid_df[[
    "day", "순수익", "승객수", "기본요금", "팁", "유류비",
    "후보2명이상_횟수", "제자리대기분", "자동이동분"
]].to_string(index=False))

# 다음 셀에서 같은 설정으로 Day 4·5를 운행할 수 있도록 저장
config_path = OUTPUT_DIR / "final_policy.json"
config_path.write_text(
    json.dumps(final_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
valid_df.to_csv(OUTPUT_DIR / "validation_day3_summary.csv", index=False)
valid_log.to_csv(OUTPUT_DIR / "validation_day3_actions.csv", index=False)

print("\n확정 정책:", final_config)
print("정책 저장:", config_path)

행동 검증 통과: 후보가 없으면 0번, 있으면 항상 1번을 선택했습니다.

[Day 3 공식 검증 결과]
 day    순수익  승객수   기본요금     팁   유류비  후보2명이상_횟수  제자리대기분  자동이동분
   3 285922   54 302400 10155 26633         19     124    105

확정 정책: {'rank_policy': '기본요금/분', 'threshold': 0, 'max_waits': 0, 'seed': 42}
정책 저장: /content/taxi_run_n4mv6549/output/final_policy.json


In [23]:

# ============================================================
# 셀 7. 확정 정책으로 Day 4·5 운행 → resultset.csv 생성
# ============================================================
import json
from google.colab import files

# 1. 셀 6에서 저장한 정책 로드
config_path = OUTPUT_DIR / "final_policy.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
seed_all(config["seed"])
print("제출 정책:", config)

# 2. 테스트 환경 로드
test_graph, test_requests = load_data(DATA_DIR, split="test")
TEST_DAYS = (4, 5)
assert set(TEST_DAYS).issubset(test_requests["day"].unique())
assert MAX_STEPS == 2000

# 3. 공식 제출 형식: 날짜별 2,000행, 총 4,000행
submission = pd.DataFrame({
    "id": np.arange(4000, dtype=np.int64),
    "day": np.repeat(TEST_DAYS, MAX_STEPS),
    "step": np.tile(np.arange(MAX_STEPS, dtype=np.int64), len(TEST_DAYS)),
    "passenger_id": np.full(4000, -1, dtype=np.int64),
})

test_summaries = []

for day_idx, day in enumerate(TEST_DAYS):
    # run_policy_day는 매 단계 step() 호출 전에 실제 승객 ID를 기록함
    summary, log = run_policy_day(
        test_graph,
        test_requests,
        day,
        make_final_policy(config),
    )

    pids = log["passenger_id"].to_numpy(dtype=np.int64)
    assert len(pids) <= MAX_STEPS
    assert np.array_equal(log["step"].to_numpy(), np.arange(len(pids)))

    # 실제 선택 승객 수와 공식 환경의 운행 완료 건수가 같아야 함
    served_ids = pids[pids != -1]
    assert len(served_ids) == len(np.unique(served_ids)) == summary["승객수"]

    # 실제 행동 구간만 입력. 남은 구간은 초기값 -1 유지
    start = day_idx * MAX_STEPS
    submission.loc[start:start + len(pids) - 1, "passenger_id"] = pids

    test_summaries.append({**summary, "실제행동수": len(pids)})

# 4. 형식 검증 후 저장
assert submission.shape == (4000, 4)
assert list(submission.columns) == ["id", "day", "step", "passenger_id"]
assert np.array_equal(submission["id"].to_numpy(), np.arange(4000))
assert not submission.isna().any().any()

for day in TEST_DAYS:
    part = submission.loc[submission["day"] == day]
    assert np.array_equal(part["step"].to_numpy(), np.arange(MAX_STEPS))

output_path = OUTPUT_DIR / "resultset.csv"
submission.to_csv(output_path, index=False)

report = pd.DataFrame(test_summaries)
print("\n[테스트 운행 결과]")
print(report[[
    "day", "순수익", "승객수", "기본요금", "팁", "유류비", "실제행동수"
]].to_string(index=False))
print(f"\nDay 4·5 합산 순수익: {report['순수익'].sum():,}원")
print(f"제출 파일: {output_path}  ({len(submission):,}행)")

# 5. Colab에서 제출 파일 다운로드
files.download(str(output_path))

제출 정책: {'rank_policy': '기본요금/분', 'threshold': 0, 'max_waits': 0, 'seed': 42}
Loaded 30 nodes, 60 edges, 800 test requests

[테스트 운행 결과]
 day    순수익  승객수   기본요금     팁   유류비  실제행동수
   4 333535   61 336600 17265 20330    227
   5 314114   58 324900 12420 23206    192

Day 4·5 합산 순수익: 647,649원
제출 파일: /content/taxi_run_n4mv6549/output/resultset.csv  (4,000행)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
### 셀 8 — ‘분당 수익률’ 대신 ‘시간 비용을 뺀 순수익’으로 승객 선택


# ============================================================
# 셀 8. 시간 비용을 반영한 승객 선택 정책 실험
# 필요: 셀 1~7에서 정의한 함수 및 학습일 결과
# ============================================================
import json
import numpy as np
import pandas as pd

# 1. Day 1·2의 기준 정책 기록에서만 시간 가치 계산
fit_summary = baseline_summary[baseline_summary["day"].isin(FIT_DAYS)]
fit_log = baseline_log[baseline_log["day"].isin(FIT_DAYS)]

fit_minutes = int(fit_log["elapsed_min"].sum())
fit_revenue = int(fit_summary["순수익"].sum())
assert fit_minutes > 0

won_per_minute = fit_revenue / fit_minutes
print(f"기준 정책: {fit_revenue:,}원 / {fit_minutes:,}분")
print(f"시간 비용 추정값: {won_per_minute:.2f}원/분")


# 2. 수락/대기는 기존과 동일. 여러 후보가 있을 때만 선택이 달라질 수 있음.
def make_time_cost_policy(rate):
    def policy(env, choices):
        if not choices:
            return 0
        if len(choices) == 1:
            return 1

        def score(choice):
            request, pickup_min, trip_min = choice
            net = estimate_candidate_net(env, request, pickup_min)
            return net - rate * (pickup_min + trip_min)

        # 동점이면 기존 공식 후보 순서 우선
        return 1 + max(range(len(choices)), key=lambda j: score(choices[j]))

    return policy


# 3. 같은 날짜에서 기존 정책과 새 정책을 비교
base_by_day = {
    int(row["day"]): row
    for row in baseline_summary.to_dict("records")
}
base_by_day[VALID_DAY] = valid_summary  # 셀 6: 공식 후보 1번 정책의 Day 3 결과

def compare_time_cost(day):
    old = base_by_day[day]
    new, log = run_policy_day(
        graph, train_requests, day, make_time_cost_policy(won_per_minute)
    )

    # 이번 실험에서는 의도적 대기가 한 번도 없어야 함
    assert not log["kind"].eq("후보거절대기").any()

    return {
        "day": day,
        "기존순수익": old["순수익"],
        "새순수익": new["순수익"],
        "순수익차이": new["순수익"] - old["순수익"],
        "승객수차이": new["승객수"] - old["승객수"],
        "기본요금차이": new["기본요금"] - old["기본요금"],
        "팁차이": new["팁"] - old["팁"],
        "유류비차이": new["유류비"] - old["유류비"],
        "1번외선택횟수": int((log["action"] > 1).sum()),
    }


rows = [compare_time_cost(day) for day in FIT_DAYS]
fit_cmp = pd.DataFrame(rows)
print("\n[Day 1·2 비교: 차이 = 새 정책 − 기존 정책]")
print(fit_cmp.to_string(index=False))

gain12 = int(fit_cmp["순수익차이"].sum())
print(f"\nDay 1·2 합산 차이: {gain12:+,}원")

# 4. 한 날짜에서의 큰 이익만으로 정책을 바꾸지 않음
fit_pass = gain12 > 0 and (fit_cmp["순수익차이"] >= 0).all()
adopt = False

if fit_pass:
    day3_row = compare_time_cost(VALID_DAY)
    rows.append(day3_row)

    print("\n[Day 3 비교 — 시간 비용은 재조정하지 않음]")
    print(pd.DataFrame([day3_row]).to_string(index=False))

    adopt = day3_row["순수익차이"] > 0
    if adopt:
        print("\n결론: Day 1·2 채택 기준을 통과했고 Day 3에서도 개선됐습니다.")
    else:
        print("\n결론: Day 3에서 개선되지 않아 기존 정책을 유지합니다.")
else:
    print("\n결론: Day 1·2 채택 기준 미충족. Day 3을 추가로 보지 않고 종료합니다.")

# 5. 실험 결과만 별도 저장. 기존 정책 및 제출 CSV는 덮어쓰지 않음.
exp_dir = OUTPUT_DIR / "time_cost_experiment"
exp_dir.mkdir(exist_ok=True)

pd.DataFrame(rows).to_csv(exp_dir / "comparison.csv", index=False)
(exp_dir / "experiment.json").write_text(
    json.dumps(
        {
            "policy": "net_minus_time_cost",
            "won_per_minute": won_per_minute,
            "fit_days": list(FIT_DAYS),
            "passed_comparison": bool(adopt),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("실험 결과 저장:", exp_dir)

기준 정책: 655,758원 / 1,800분
시간 비용 추정값: 364.31원/분

[Day 1·2 비교: 차이 = 새 정책 − 기존 정책]
 day  기존순수익   새순수익  순수익차이  승객수차이  기본요금차이  팁차이  유류비차이  1번외선택횟수
   1 332727 329878  -2849     -6   -5700  990  -1861        6
   2 323031 302446 -20585     -7  -21000  855    440        2

Day 1·2 합산 차이: -23,434원

결론: Day 1·2 채택 기준 미충족. Day 3을 추가로 보지 않고 종료합니다.
실험 결과 저장: /content/taxi_run_n4mv6549/output/time_cost_experiment


In [25]:
# ============================================================
# 셀 9. 공식 1번 우선 + 근소한 차이에서만 하차 위치로 재선택
# 기존 제출 파일, 기존 정책, 공식 환경은 수정하지 않음
# ============================================================
from functools import lru_cache
import pandas as pd

NEAR_RATIO = 0.95
PICKUP_RADIUS_MIN = 5


def make_reach_policy(ref_graph):
    # 공식 경로 시간은 같은 출발 노드·도착 노드·시간대이면 동일함.
    # 요청 분포를 사용하지 않고 도로망의 도달 범위만 계산.
    route_env = TaxiEnv(ref_graph, train_requests, day=FIT_DAYS[0])
    nodes = tuple(int(n) for n in ref_graph.nodes)
    switches = []

    @lru_cache(maxsize=None)
    def reachable_count(pos, hour):
        count = 0
        for node in nodes:
            if route_env.hops[pos][node] > 3:
                continue
            _, minutes = route_env.shortest_route(pos, node, hour * 60)
            if minutes <= PICKUP_RADIUS_MIN:
                count += 1
        return count

    def policy(env, choices):
        if not choices:
            return 0
        if len(choices) == 1:
            return 1

        def fare_rate(choice):
            r, p, d = choice
            return r["base_fare"] / max(1, p + d)

        def reach_after(choice):
            r, p, d = choice
            arrival = env.time + p + d
            # 영업 종료 시각이면 다음 승객을 잡을 수 없음
            if arrival >= END:
                return 0
            return reachable_count(int(r["dropoff"]), int(arrival // 60))

        first_rate = fare_rate(choices[0])

        # 충분히 가까운 대안이 없다면 경로 계산 없이 원래 행동 유지
        close = [
            i for i in range(1, len(choices))
            if fare_rate(choices[i]) >= NEAR_RATIO * first_rate
        ]
        if not close:
            return 1

        first_reach = reach_after(choices[0])
        best_idx, best_reach = 0, first_reach

        # choices 자체가 공식 수익률 내림차순이므로,
        # 도달 범위가 같으면 수익률이 높은 기존 후보를 유지
        for i in close:
            reach = reach_after(choices[i])
            if reach > best_reach:
                best_idx, best_reach = i, reach

        if best_idx != 0:
            switches.append({
                "day": env.day,
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "선택순번": best_idx + 1,
                "1번하차": int(choices[0][0]["dropoff"]),
                "선택하차": int(choices[best_idx][0]["dropoff"]),
                "수익률비율": round(fare_rate(choices[best_idx]) / first_rate, 3),
                "1번도달노드수": first_reach,
                "선택도달노드수": best_reach,
            })

        return best_idx + 1

    return policy, switches


rows = []
all_switches = []
base_by_day = baseline_summary.set_index("day")

for day in FIT_DAYS:
    policy, switches = make_reach_policy(graph)
    summary, log = run_policy_day(graph, train_requests, day, policy)
    base = base_by_day.loc[day]

    assert not log["kind"].eq("후보거절대기").any()
    assert len(switches) == int((log["action"] > 1).sum())

    rows.append({
        "day": day,
        "기존순수익": int(base["순수익"]),
        "새순수익": summary["순수익"],
        "순수익차이": summary["순수익"] - int(base["순수익"]),
        "승객수차이": summary["승객수"] - int(base["승객수"]),
        "픽업이동분차이": summary["픽업이동분"] - int(base["픽업이동분"]),
        "자동이동분차이": summary["자동이동분"] - int(base["자동이동분"]),
        "선택변경횟수": len(switches),
    })
    all_switches.extend(switches)

reach_cmp = pd.DataFrame(rows)
print("[Day 1·2 결과 | 차이 = 새 정책 − 기존 정책]")
print(reach_cmp.to_string(index=False))
print(f"\n합산 순수익 차이: {int(reach_cmp['순수익차이'].sum()):+,}원")

if all_switches:
    print("\n[공식 1번 대신 다른 후보를 선택한 실제 상황]")
    print(pd.DataFrame(all_switches).to_string(index=False))
else:
    print("\n두 날 모두 선택 변경 없음: 이 규칙은 기준 정책과 같은 경로를 운행했습니다.")

reach_pass = (
    reach_cmp["순수익차이"].ge(0).all()
    and reach_cmp["순수익차이"].sum() > 0
)
print("\n판정:", "Day 3 확인 대상으로 유지" if reach_pass else "기존 정책 유지")

exp9_dir = OUTPUT_DIR / "reach_tiebreak_experiment"
exp9_dir.mkdir(exist_ok=True)
reach_cmp.to_csv(exp9_dir / "comparison.csv", index=False)
if all_switches:
    pd.DataFrame(all_switches).to_csv(exp9_dir / "switches.csv", index=False)

[Day 1·2 결과 | 차이 = 새 정책 − 기존 정책]
 day  기존순수익   새순수익  순수익차이  승객수차이  픽업이동분차이  자동이동분차이  선택변경횟수
   1 332727 329712  -3015     -2        8      -20       4
   2 323031 315314  -7717     -1       12        5       4

합산 순수익 차이: -10,732원

[공식 1번 대신 다른 후보를 선택한 실제 상황]
 day    시각  선택순번  1번하차  선택하차  수익률비율  1번도달노드수  선택도달노드수
   1 07:07     2    11    16  0.982        7        8
   1 11:49     2    19     0  0.983       12       13
   1 12:36     2    15    21  0.952        5        8
   1 14:31     2     1    27  1.000       11       14
   2 08:26     2     6    28  1.000        5        7
   2 10:23     2    18     7  0.976        9       15
   2 11:41     2     4    16  0.995        9       10
   2 13:44     2     5    26  0.972        5       10

판정: 기존 정책 유지


In [26]:
# ============================================================
# 셀 10. 현재 보이는 후보만 사용한 2건 연속 운행 실험
# Day 1·2만 사용. 기존 정책/환경/제출 파일 변경 없음.
# ============================================================
def make_visible_pair_policy():
    stats = {
        "복수후보상태": 0,
        "연속운행가능상태": 0,
        "선택변경": 0,
    }
    decisions = []

    def policy(env, choices):
        if not choices:
            return 0
        if len(choices) == 1:
            return 1

        stats["복수후보상태"] += 1

        # 기준: 공식 1번 승객을 단독 운행할 때의 기본요금/분
        first, p0, d0 = choices[0]
        baseline_rate = first["base_fare"] / max(1, p0 + d0)
        best_rate = baseline_rate
        best_action = 1
        best_pair = None
        found_pair = False

        # 현재 보이는 승객 i를 먼저 태우고,
        # 하차 후에도 현재 후보 j의 픽업 기한이 남아 있는지 확인
        for i, (ri, pi, di) in enumerate(choices):
            after_i = env.time + pi + di
            pos_i = ri["dropoff"]

            for j, (rj, _, _) in enumerate(choices):
                if i == j:
                    continue

                deadline_j = rj["minute"] + rj["max_wait"]

                # 공식 환경과 동일: 기한과 같은 시각에 픽업해도 만료
                if after_i >= deadline_j:
                    continue
                if env.hops[pos_i][rj["pickup"]] > 3:
                    continue

                _, pj = env.shortest_route(pos_i, rj["pickup"], after_i)
                pickup_j = after_i + pj
                if pickup_j >= deadline_j:
                    continue

                _, dj = env.shortest_route(rj["pickup"], rj["dropoff"], pickup_j)
                finish_j = pickup_j + dj
                if finish_j > END:
                    continue

                found_pair = True
                total_minutes = max(1, finish_j - env.time)
                pair_rate = (ri["base_fare"] + rj["base_fare"]) / total_minutes

                # 공식 후보 순서대로 탐색. 동점이면 먼저 찾은 순서를 유지.
                if pair_rate > best_rate + 1e-9:
                    best_rate = pair_rate
                    best_action = i + 1
                    best_pair = (ri, rj, pj, deadline_j - pickup_j)

        stats["연속운행가능상태"] += int(found_pair)

        if best_action != 1:
            stats["선택변경"] += 1
            ri, rj, second_pickup_min, slack = best_pair
            decisions.append({
                "day": env.day,
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "선택순번": best_action,
                "첫승객ID": int(ri["request_id"]),
                "예상둘째ID": int(rj["request_id"]),
                "둘째픽업이동분": int(second_pickup_min),
                "둘째픽업기한여유분": int(slack),
                "기준요금분": round(baseline_rate, 1),
                "연속요금분": round(best_rate, 1),
            })

        return best_action

    return policy, stats, decisions


rows, all_decisions = [], []
base_by_day = baseline_summary.set_index("day")

for day in FIT_DAYS:
    policy, stats, decisions = make_visible_pair_policy()
    summary, log = run_policy_day(graph, train_requests, day, policy)
    base = base_by_day.loc[day]

    # 예상한 두 번째 승객을 실제 다음 행동에서도 태웠는지 확인.
    # 미래 요청을 정책 입력으로 쓰는 것이 아니라, 운행 후 로그만 평가.
    selected = log["passenger_id"].to_numpy()
    next_pid = {
        int(selected[k]): int(selected[k + 1])
        for k in range(len(selected) - 1)
        if selected[k] != -1
    }
    for x in decisions:
        x["예상연속운행실현"] = (
            next_pid.get(x["첫승객ID"], -1) == x["예상둘째ID"]
        )

    rows.append({
        "day": day,
        "기존순수익": int(base["순수익"]),
        "새순수익": summary["순수익"],
        "순수익차이": summary["순수익"] - int(base["순수익"]),
        "승객수차이": summary["승객수"] - int(base["승객수"]),
        **stats,
        "변경후예상실현": sum(x["예상연속운행실현"] for x in decisions),
    })
    all_decisions.extend(decisions)

pair_cmp = pd.DataFrame(rows)
print("[Day 1·2: 현재 후보 2건 연속 운행 정책]")
print(pair_cmp.to_string(index=False))
print(f"\n합산 순수익 차이: {int(pair_cmp['순수익차이'].sum()):+,}원")

if all_decisions:
    print("\n[실제로 공식 1번 대신 다른 승객을 선택한 상황]")
    print(pd.DataFrame(all_decisions).to_string(index=False))
else:
    print("\n기존 정책과 다른 행동을 선택한 경우가 없습니다.")

pair_pass = (
    pair_cmp["순수익차이"].ge(0).all()
    and pair_cmp["순수익차이"].sum() > 0
)
print("\n판정:", "Day 3 검증 대상으로 유지" if pair_pass else "기존 정책 유지")

pair_dir = OUTPUT_DIR / "visible_pair_experiment"
pair_dir.mkdir(exist_ok=True)
pair_cmp.to_csv(pair_dir / "comparison.csv", index=False)
if all_decisions:
    pd.DataFrame(all_decisions).to_csv(pair_dir / "decisions.csv", index=False)

[Day 1·2: 현재 후보 2건 연속 운행 정책]
 day  기존순수익   새순수익  순수익차이  승객수차이  복수후보상태  연속운행가능상태  선택변경  변경후예상실현
   1 332727 332727      0      0      31         7     0        0
   2 323031 323031      0      0      29         5     0        0

합산 순수익 차이: +0원

기존 정책과 다른 행동을 선택한 경우가 없습니다.

판정: 기존 정책 유지


In [27]:
# ============================================================
# 셀 11. 한 번의 선택 변경이 하루 순수익에 미치는 효과를 학습
# Day 1·2만 사용 / 공식 환경 수정 없음 / 테스트 요청 사용 안 함
# ============================================================
import numpy as np
import pandas as pd

exp11_dir = OUTPUT_DIR / "one_step_advantage"
exp11_dir.mkdir(exist_ok=True)

# 현재 시점에서 관측 가능한 후보 1번과 2번의 차이만 입력한다.
# request_id, 날짜, 행동 순번은 모델 특성으로 쓰지 않는다.
def pair_features(env, choices):
    def features(choice):
        r, p, d = choice
        duration = max(1, p + d)
        fare = float(r["base_fare"])
        net = float(estimate_candidate_net(env, r, p))
        return np.array([
            fare / 10000.0,
            net / 10000.0,
            (fare / duration) / 500.0,
            (net / duration) / 500.0,
            p / 10.0,
            d / 20.0,
        ], dtype=np.float64)

    f1 = features(choices[0])
    f2 = features(choices[1])
    return f2 - f1


# 1. 기준 정책이 방문한 복수 후보 상태의 입력 특성 수집
contexts = {}
for day in FIT_DAYS:
    state_rows = {}
    counter = [0]

    def record_reference(env, choices):
        step = counter[0]
        counter[0] += 1
        if len(choices) >= 2:
            state_rows[step] = pair_features(env, choices)
        return 1 if choices else 0

    ref_summary, _ = run_policy_day(graph, train_requests, day, record_reference)
    reference_net = int(
        baseline_summary.loc[baseline_summary["day"] == day, "순수익"].iloc[0]
    )
    assert ref_summary["순수익"] == reference_net
    contexts[day] = state_rows

print("수집한 복수 후보 상태:", {d: len(x) for d, x in contexts.items()})


# 2. 지정한 단계에서만 2번 선택. 다른 모든 단계에서는 기존 정책 사용.
# 매번 공식 환경을 처음부터 실행하므로 내부 상태를 복사하거나 조작하지 않는다.
def run_single_change(day, change_step):
    env = TaxiEnv(graph, train_requests, day)
    changed = False

    for step in range(MAX_STEPS):
        choices = env.candidates()

        if step == change_step:
            assert len(choices) >= 2
            action = 2
            changed = True
        else:
            action = 1 if choices else 0

        _, _, done, _ = env.step(action)
        if done:
            assert changed
            return int(env.earned)

    raise RuntimeError(f"Day {day}: MAX_STEPS 초과")


samples = {}
diagnostics = []

for day in FIT_DAYS:
    ref_net = int(
        baseline_summary.loc[baseline_summary["day"] == day, "순수익"].iloc[0]
    )
    xs, ys = [], []

    for step, x in contexts[day].items():
        delta = run_single_change(day, step) - ref_net
        xs.append(x)
        ys.append(delta)
        diagnostics.append({
            "day": day,
            "step": step,
            "2번선택_하루순수익차이": delta,
        })

    samples[day] = (np.stack(xs), np.asarray(ys, dtype=float))

diag = pd.DataFrame(diagnostics)
print("\n[한 단계에서만 2번 선택한 결과]")
for day in FIT_DAYS:
    y = samples[day][1]
    print(
        f"Day {day}: 전체 {len(y)}건 | "
        f"이득 {int((y > 0).sum())}건 | 손해 {int((y < 0).sum())}건 | "
        f"동일 {int((y == 0).sum())}건 | 중앙값 {np.median(y):+,.0f}원"
    )

diag.to_csv(exp11_dir / "single_change_outcomes.csv", index=False)


# 3. 작은 Ridge 모델: 입력 특성의 평균·표준편차도 학습일에서만 계산
# 먼 미래의 우연한 경로 차이가 모델을 지배하지 않도록 학습값만 제한.
# 평가에는 제한하지 않은 공식 순수익을 그대로 사용.
def fit_advantage(train_days):
    X = np.concatenate([samples[d][0] for d in train_days], axis=0)
    y = np.concatenate([samples[d][1] for d in train_days], axis=0)

    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd < 1e-6] = 1.0
    Z = (X - mu) / sd

    target = np.clip(y, -10000.0, 10000.0)
    bias = float(target.mean())
    ridge = 20.0
    weights = np.linalg.solve(
        Z.T @ Z + ridge * np.eye(Z.shape[1]),
        Z.T @ (target - bias),
    )
    return mu, sd, weights, bias


def make_advantage_policy(model):
    mu, sd, weights, bias = model

    def policy(env, choices):
        if not choices:
            return 0
        if len(choices) == 1:
            return 1

        x = pair_features(env, choices)
        predicted_gain = float(((x - mu) / sd) @ weights + bias)

        # 표본이 작으므로 근소한 예측 이득만으로는 기준 행동을 바꾸지 않음.
        return 2 if predicted_gain > 2000.0 else 1

    return policy


# 4. 날짜를 교차해 실제 운행으로 평가
# '예측값의 정확도'가 아니라 정책을 하루 돌렸을 때의 순수익을 측정한다.
cv_rows = []
for train_day, valid_day in [(1, 2), (2, 1)]:
    model = fit_advantage([train_day])
    policy = make_advantage_policy(model)
    summary, log = run_policy_day(graph, train_requests, valid_day, policy)

    ref_net = int(
        baseline_summary.loc[
            baseline_summary["day"] == valid_day, "순수익"
        ].iloc[0]
    )
    cv_rows.append({
        "학습일": train_day,
        "평가일": valid_day,
        "기준순수익": ref_net,
        "모델순수익": summary["순수익"],
        "차이": summary["순수익"] - ref_net,
        "2번선택횟수": int(log["action"].eq(2).sum()),
    })

adv_cv = pd.DataFrame(cv_rows)
print("\n[다른 날짜에서의 실제 운행 결과]")
print(adv_cv.to_string(index=False))
print(f"\n교차평가 합산 차이: {int(adv_cv['차이'].sum()):+,}원")

adv_pass = adv_cv["차이"].ge(0).all() and adv_cv["차이"].sum() > 0
print("판정:", "Day 3 확인 대상으로 유지" if adv_pass else "기존 정책 유지")
adv_cv.to_csv(exp11_dir / "cross_day_comparison.csv", index=False)

수집한 복수 후보 상태: {1: 31, 2: 29}

[한 단계에서만 2번 선택한 결과]
Day 1: 전체 31건 | 이득 9건 | 손해 22건 | 동일 0건 | 중앙값 -2,077원
Day 2: 전체 29건 | 이득 12건 | 손해 16건 | 동일 1건 | 중앙값 -661원

[다른 날짜에서의 실제 운행 결과]
 학습일  평가일  기준순수익  모델순수익     차이  2번선택횟수
   1    2 323031 323031      0       0
   2    1 332727 318254 -14473       1

교차평가 합산 차이: -14,473원
판정: 기존 정책 유지


In [28]:
# ============================================================
# 셀 12. 후보 1명 상태에서 '딱 한 번의 1분 대기' 효과 측정
# Day 1·2만 사용 / 테스트 미사용 / 제출 파일 변경 없음
# ============================================================
wait_exp_dir = OUTPUT_DIR / "single_wait_diagnostic"
wait_exp_dir.mkdir(exist_ok=True)


def measure_one_wait(day, target_step, expected_pid):
    env = TaxiEnv(graph, train_requests, day)
    record = None

    for step in range(MAX_STEPS):
        choices = env.candidates()

        if step == target_step:
            # 여기까지는 기준 정책과 똑같이 운행했어야 함
            assert len(choices) == 1, (day, step, len(choices))
            r, p, d = choices[0]
            pid = int(r["request_id"])
            assert pid == expected_pid, (day, step, pid, expected_pid)

            deadline = int(r["minute"] + r["max_wait"])
            record = {
                "day": day,
                "step": step,
                "minute": int(env.time),
                "pid": pid,
                "픽업분": int(p),
                "운행분": int(d),
                "예상순수익": int(estimate_candidate_net(env, r, p)),
                # 1이면 같은 이동 시간이 유지될 때 1분 뒤에는 픽업 기한 초과
                "현재픽업여유분": int(deadline - env.time - p),
            }

            # 후보가 있지만 이 단계에서만 대기
            action = 0
            before_time, before_pos = env.time, env.pos
        else:
            # 바로 다음 단계부터는 다시 기준 정책
            action = 1 if choices else 0

            if record is not None and step == target_step + 1:
                record["다음행동"] = (
                    "후보 없음" if not choices
                    else "같은 승객 수락"
                    if int(choices[0][0]["request_id"]) == expected_pid
                    else "다른 승객 수락"
                )

        _, _, done, _ = env.step(action)

        if step == target_step:
            assert env.time == before_time + 1 and env.pos == before_pos

        if done:
            assert record is not None
            record.setdefault("다음행동", "영업 종료")
            record["대기후_하루순수익"] = int(env.earned)
            record["대기후_승객수"] = int(env.served)
            return record

    raise RuntimeError(f"Day {day}: MAX_STEPS 초과")


# 기준 정책이 후보 한 명을 즉시 태웠던 단계만 사용
targets = baseline_log.loc[
    baseline_log["day"].isin(FIT_DAYS)
    & baseline_log["candidate_count"].eq(1)
    & baseline_log["action"].eq(1),
    ["day", "step", "passenger_id"],
]

ref = baseline_summary.set_index("day")
records = []

for row in targets.itertuples(index=False):
    day = int(row.day)
    rec = measure_one_wait(day, int(row.step), int(row.passenger_id))
    rec["순수익차이"] = rec["대기후_하루순수익"] - int(ref.loc[day, "순수익"])
    rec["승객수차이"] = rec["대기후_승객수"] - int(ref.loc[day, "승객수"])
    records.append(rec)

single_wait = pd.DataFrame(records)

print("[1] 각 상황에서 한 번만 기다렸을 때의 결과")
by_day = single_wait.groupby("day")["순수익차이"].agg(
    실험수="size",
    이득건수=lambda s: int((s > 0).sum()),
    손해건수=lambda s: int((s < 0).sum()),
    동일건수=lambda s: int((s == 0).sum()),
    중앙값="median",
)
print(by_day.to_string())

print("\n[2] 1분 후 실제로 무엇이 달라졌는가?")
by_next = single_wait.groupby(["day", "다음행동"])["순수익차이"].agg(
    건수="size",
    이득건수=lambda s: int((s > 0).sum()),
    중앙값="median",
)
print(by_next.to_string())

# 대기 '전'에 알 수 있는 정보로도 방향이 구분되는지 확인
single_wait["픽업여유구간"] = pd.cut(
    single_wait["현재픽업여유분"],
    bins=[0, 1, 3, np.inf],
    labels=["1분", "2~3분", "4분 이상"],
)
print("\n[3] 대기 전에 관측한 픽업 여유별 결과")
by_slack = single_wait.groupby(
    ["day", "픽업여유구간"], observed=True
)["순수익차이"].agg(
    건수="size",
    이득건수=lambda s: int((s > 0).sum()),
    중앙값="median",
)
print(by_slack.to_string())

single_wait.to_csv(wait_exp_dir / "single_wait_outcomes.csv", index=False)
print("\n저장:", wait_exp_dir / "single_wait_outcomes.csv")

[1] 각 상황에서 한 번만 기다렸을 때의 결과
     실험수  이득건수  손해건수  동일건수    중앙값
day                              
1     34    11    20     3 -595.5
2     34    19     9     6  204.5

[2] 1분 후 실제로 무엇이 달라졌는가?
              건수  이득건수      중앙값
day 다음행동                       
1   같은 승객 수락  24     8   -202.5
    다른 승객 수락   5     3    442.0
    후보 없음      5     0 -11222.0
2   같은 승객 수락  24    12     65.0
    다른 승객 수락   7     5   3621.0
    후보 없음      3     2   7817.0

[3] 대기 전에 관측한 픽업 여유별 결과
            건수  이득건수      중앙값
day 픽업여유구간                   
1   1분       5     0 -11222.0
    2~3분    11     3  -1596.0
    4분 이상   18     8      0.0
2   1분       5     3   7817.0
    2~3분    13     8   1142.0
    4분 이상   16     8     65.0

저장: /content/taxi_run_n4mv6549/output/single_wait_diagnostic/single_wait_outcomes.csv


In [29]:
# ============================================================
# 셀 13. 실제 반사실 운행 결과로 '하루 한 번 대기' 정책 학습
# 전제: 셀 12까지 실행 완료
# ============================================================
import numpy as np
import pandas as pd

exp13_dir = OUTPUT_DIR / "learned_single_wait"
exp13_dir.mkdir(exist_ok=True)

# 1. 셀 12의 학습값에 결정 당시에 알 수 있던 위치 정보만 결합
# 다음행동, 대기후_하루순수익 등 사후 정보는 모델 입력에 사용하지 않음
outcomes = pd.read_csv(
    OUTPUT_DIR / "single_wait_diagnostic" / "single_wait_outcomes.csv"
)

state_locations = baseline_log.loc[
    baseline_log["candidate_count"].eq(1),
    ["day", "step", "passenger_id", "start_node"],
].rename(columns={"passenger_id": "pid", "start_node": "taxi_node"})

pickup_lookup = train_requests[
    ["day", "request_id", "pickup"]
].rename(columns={"request_id": "pid", "pickup": "pickup_node"})

examples = (
    outcomes[["day", "step", "pid", "minute", "현재픽업여유분", "순수익차이"]]
    .merge(
        state_locations,
        on=["day", "step", "pid"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        pickup_lookup,
        on=["day", "pid"],
        how="left",
        validate="many_to_one",
    )
)

assert len(examples) == len(outcomes)
assert not examples.isna().any().any()

# 고정 도로망의 hop 거리. 승객 요청이나 미래 정보와 무관.
hop_dist = dict(nx.all_pairs_shortest_path_length(graph))


# 2. 비슷한 과거 사례로 1분 대기의 예상 이득을 계산
def fit_wait_estimator(fit_days):
    data = examples.loc[examples["day"].isin(fit_days)].copy()
    if data.empty:
        raise ValueError("학습 사례가 없습니다.")

    minutes = data["minute"].to_numpy(dtype=float)
    taxi_nodes = data["taxi_node"].to_numpy(dtype=int)
    pickup_nodes = data["pickup_node"].to_numpy(dtype=int)
    slack_values = data["현재픽업여유분"].to_numpy(dtype=float)

    # 소수의 큰 경로 차이가 예측을 지배하지 않게 학습값만 제한.
    # 최종 평가는 자르지 않은 공식 순수익으로 진행.
    gains = np.clip(data["순수익차이"].to_numpy(dtype=float), -10000, 10000)

    def predict(minute, taxi_node, pickup_node, slack):
        time_gap = np.abs(minutes - minute)
        taxi_gap = np.array([hop_dist[taxi_node][int(n)] for n in taxi_nodes])
        pickup_gap = np.array([hop_dist[pickup_node][int(n)] for n in pickup_nodes])
        slack_gap = np.abs(slack_values - slack)

        # 위치는 숫자 ID의 차가 아니라 실제 도로망의 hop 거리로 비교
        weights = np.exp(
            -time_gap / 120.0
            -taxi_gap / 2.0
            -pickup_gap / 2.0
            -slack_gap / 4.0
        )

        mass = float(weights.sum())
        # 근거가 약하면 '대기 이득 0원' 쪽으로 수축
        estimated_gain = float(np.dot(weights, gains) / (mass + 2.0))
        return estimated_gain, mass

    return predict


# 3. 정책: 후보 1명일 때만, 충분한 근거가 있으면 하루 한 번 대기
# 아래 기준은 두 평가일 결과를 보기 전에 고정하고 이 셀에서 탐색하지 않음.
MIN_WEIGHT = 1.0
MIN_GAIN = 500.0

def make_one_wait_policy(predict):
    used_wait = False
    decisions = []

    def policy(env, choices):
        nonlocal used_wait

        if not choices:
            return 0

        # 한 번 기다린 이후 또는 후보가 여러 명일 때는 공식 1번 정책 유지
        if used_wait or len(choices) != 1:
            return 1

        r, pickup_min, _ = choices[0]
        slack = int(r["minute"] + r["max_wait"] - env.time - pickup_min)

        gain, mass = predict(
            minute=int(env.time),
            taxi_node=int(env.pos),
            pickup_node=int(r["pickup"]),
            slack=slack,
        )

        if mass >= MIN_WEIGHT and gain >= MIN_GAIN:
            used_wait = True
            decisions.append({
                "minute": int(env.time),
                "pid": int(r["request_id"]),
                "예상이득": round(gain),
                "유사사례가중치합": round(mass, 2),
            })
            return 0

        return 1

    return policy, decisions


# 4. 날짜 교차평가: 평가일의 순수익차이는 모델 학습에 넣지 않음
ref_by_day = baseline_summary.set_index("day")["순수익"].to_dict()
cv_rows = []

for fit_day, eval_day in [(1, 2), (2, 1)]:
    predictor = fit_wait_estimator([fit_day])
    policy, decisions = make_one_wait_policy(predictor)

    summary, log = run_policy_day(graph, train_requests, eval_day, policy)
    waits = log.loc[log["kind"].eq("후보거절대기")]
    assert len(waits) == len(decisions) <= 1

    ref_net = int(ref_by_day[eval_day])
    actual_gain = summary["순수익"] - ref_net

    # 셀 12의 독립 시뮬레이션과 일치하는지 사후 검증.
    # 이 값은 행동 결정에는 사용되지 않음.
    if decisions:
        wait_step = int(waits.iloc[0]["step"])
        matched = examples.loc[
            examples["day"].eq(eval_day) & examples["step"].eq(wait_step),
            "순수익차이",
        ]
        assert len(matched) == 1
        assert actual_gain == int(matched.iloc[0])
        detail = decisions[0]
        clock = f"{detail['minute'] // 60:02d}:{detail['minute'] % 60:02d}"
    else:
        assert actual_gain == 0
        detail = {}
        clock = "-"

    cv_rows.append({
        "학습일": fit_day,
        "평가일": eval_day,
        "기준순수익": ref_net,
        "모델순수익": summary["순수익"],
        "실제차이": actual_gain,
        "대기횟수": len(decisions),
        "대기시각": clock,
        "예상이득": detail.get("예상이득"),
        "가중치합": detail.get("유사사례가중치합"),
    })

wait_cv = pd.DataFrame(cv_rows)
print("[하루 한 번 대기 정책: 날짜 교차평가]")
print(wait_cv.to_string(index=False))
print(f"\n합산 순수익 차이: {int(wait_cv['실제차이'].sum()):+,}원")

wait_pass = (
    wait_cv["실제차이"].ge(0).all()
    and wait_cv["실제차이"].sum() > 0
)

if wait_pass:
    print("판정: Day 3 추가 확인 대상으로 유지합니다. 아직 제출 정책은 바꾸지 않습니다.")
elif wait_cv["대기횟수"].sum() == 0:
    print("판정: 다른 날짜에서 대기를 권할 근거가 부족했습니다. 기존 정책을 유지합니다.")
else:
    print("판정: 대기 시점을 다른 날짜에서 안정적으로 고르지 못했습니다. 기존 정책을 유지합니다.")

wait_cv.to_csv(exp13_dir / "cross_day_comparison.csv", index=False)

[하루 한 번 대기 정책: 날짜 교차평가]
 학습일  평가일  기준순수익  모델순수익   실제차이  대기횟수  대기시각   예상이득  가중치합
   1    2 323031 323031      0     0     -    NaN   NaN
   2    1 332727 321432 -11295     1 10:50 1433.0   1.0

합산 순수익 차이: -11,295원
판정: 대기 시점을 다른 날짜에서 안정적으로 고르지 못했습니다. 기존 정책을 유지합니다.


In [30]:
# ============================================================
# 셀 14. 공식 1번 정책의 안전한 교체 조건 확인
# 학습/검증일만 사용. 기존 제출 CSV는 변경하지 않음.
# ============================================================
def make_safe_switch_policy():
    switches = []

    def policy(env, choices):
        if not choices:
            return 0
        if len(choices) == 1:
            return 1

        r0, p0, d0 = choices[0]
        end0 = env.time + p0 + d0
        expiry0 = r0["minute"] + r0["max_wait"]
        net0 = estimate_candidate_net(env, r0, p0)

        best_action = 1
        best_net = net0
        best_reason = None

        for action, (r, p, d) in enumerate(choices[1:], start=2):
            end = env.time + p + d

            # 조건 A: 어느 쪽을 선택하든 바로 영업 종료
            both_terminal = (end == END and end0 == END)

            # 조건 B: 다음 위치·시각이 같고 선택하지 않은 요청도 만료
            same_future = (
                end == end0
                and r["dropoff"] == r0["dropoff"]
                and expiry0 <= end
                and r["minute"] + r["max_wait"] <= end
            )

            if not (both_terminal or same_future):
                continue

            net = estimate_candidate_net(env, r, p)
            if net > best_net:
                best_action = action
                best_net = net
                best_reason = "동시 영업종료" if both_terminal else "동일 후속상태"

        if best_action != 1:
            switches.append({
                "day": env.day,
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "선택순번": best_action,
                "교체이유": best_reason,
                "확정이득": int(best_net - net0),
            })

        return best_action

    return policy, switches


# 기존 셀에서 실제로 측정한 공식 1번 정책의 날짜별 순수익
reference = {
    int(row["day"]): int(row["순수익"])
    for _, row in baseline_summary.iterrows()
}
reference[VALID_DAY] = int(valid_summary["순수익"])

rows, switch_details = [], []

for day in (*FIT_DAYS, VALID_DAY):
    policy, switches = make_safe_switch_policy()
    summary, _ = run_policy_day(graph, train_requests, day, policy)

    actual_gain = int(summary["순수익"]) - reference[day]
    guaranteed_gain = sum(s["확정이득"] for s in switches)

    # 핵심 검증: 이번에 더 번 금액 외에는 하루 총수익 차이가 없어야 함
    assert actual_gain == guaranteed_gain, (
        f"Day {day}: 실제 차이 {actual_gain}원, 교체 이득 합 {guaranteed_gain}원"
    )

    rows.append({
        "day": day,
        "기존순수익": reference[day],
        "새순수익": summary["순수익"],
        "교체횟수": len(switches),
        "순수익차이": actual_gain,
    })
    switch_details.extend(switches)

safe_result = pd.DataFrame(rows)
print("[안전한 승객 교체 결과]")
print(safe_result.to_string(index=False))

if switch_details:
    print("\n[교체 상세]")
    print(pd.DataFrame(switch_details).to_string(index=False))
else:
    print("\nDay 1~3에서는 안전하게 이득을 확보할 교체 기회가 없었습니다.")

print("\n기존 647,649원 제출 파일은 변경하지 않았습니다.")

[안전한 승객 교체 결과]
 day  기존순수익   새순수익  교체횟수  순수익차이
   1 332727 332727     0      0
   2 323031 323031     0      0
   3 285922 285922     0      0

Day 1~3에서는 안전하게 이득을 확보할 교체 기회가 없었습니다.

기존 647,649원 제출 파일은 변경하지 않았습니다.


In [31]:
# ============================================================
# 셀 15. 셀 14의 고정된 정책을 테스트에 적용
# 기존 resultset.csv는 읽기만 하고 절대 덮어쓰지 않음
# ============================================================
old_path = OUTPUT_DIR / "resultset.csv"
old_sub = pd.read_csv(old_path)

assert old_sub.shape == (4000, 4)
assert list(old_sub.columns) == ["id", "day", "step", "passenger_id"]

# 셀 7에서 실제로 확인한 기존 날짜별 공식 순수익
old_net = {4: 333_535, 5: 314_114}

# 제출 형식만 동일하게 사용. 행동은 아래 공식 환경 운행으로 다시 생성
new_sub = old_sub[["id", "day", "step"]].copy()
new_sub["passenger_id"] = np.int64(-1)

rows = []
all_switches = []

for day in (4, 5):
    policy, switches = make_safe_switch_policy()
    summary, log = run_policy_day(test_graph, test_requests, day, policy)

    day_rows = new_sub.index[new_sub["day"].eq(day)]
    assert len(day_rows) == MAX_STEPS
    assert np.array_equal(
        new_sub.loc[day_rows, "step"].to_numpy(),
        np.arange(MAX_STEPS),
    )

    pids = log["passenger_id"].to_numpy(dtype=np.int64)
    new_sub.loc[day_rows[:len(pids)], "passenger_id"] = pids

    actual_gain = int(summary["순수익"]) - old_net[day]
    switch_gain = sum(s["확정이득"] for s in switches)

    if actual_gain != switch_gain:
        raise RuntimeError(
            f"Day {day}: 기존 제출 대비 차이 {actual_gain:+,}원과 "
            f"교체 이득 합 {switch_gain:+,}원이 다릅니다. "
            "새 제출 파일을 저장하지 않습니다."
        )

    # 교체가 없다면 점수뿐 아니라 모든 제출 행동도 같아야 함
    if not switches:
        assert np.array_equal(
            new_sub.loc[day_rows, "passenger_id"].to_numpy(),
            old_sub.loc[day_rows, "passenger_id"].to_numpy(),
        ), f"Day {day}: 교체가 없는데 기존 제출과 행동이 다릅니다."

    all_switches.extend(switches)
    rows.append({
        "day": day,
        "기존순수익": old_net[day],
        "새순수익": int(summary["순수익"]),
        "교체횟수": len(switches),
        "확인이득": actual_gain,
    })

report = pd.DataFrame(rows)
total_gain = int(report["확인이득"].sum())

print(report.to_string(index=False))
print(f"\n합산 순수익: {report['새순수익'].sum():,}원")
print(f"기존 제출 대비: {total_gain:+,}원")

if total_gain > 0:
    print("\n[실제 교체 내역]")
    print(pd.DataFrame(all_switches).to_string(index=False))

    safe_dir = OUTPUT_DIR / "safe_switch_submission"
    safe_dir.mkdir(exist_ok=True)
    safe_path = safe_dir / "resultset.csv"
    new_sub.to_csv(safe_path, index=False)
    print("새 제출 파일:", safe_path)

    from google.colab import files
    files.download(str(safe_path))
else:
    print("\n행동과 점수가 기존 제출과 동일합니다. 새로 제출할 필요가 없습니다.")

 day  기존순수익   새순수익  교체횟수  확인이득
   4 333535 334135     1   600
   5 314114 314114     0     0

합산 순수익: 648,249원
기존 제출 대비: +600원

[실제 교체 내역]
 day    시각  선택순번    교체이유  확정이득
   4 10:03     2 동일 후속상태   600
새 제출 파일: /content/taxi_run_n4mv6549/output/safe_switch_submission/resultset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
from google.colab import files

file_path = "/content/taxi_run_n4mv6549/output/safe_switch_submission/resultset.csv"

files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
# ============================================================
# 셀 16. 마감 직전의 승객 선택만 변경하는 실험
# 비교 기준: 현재 최고 제출에 사용한 셀 14의 안전 교체 정책
# Day 1·2만 사용 / 테스트 미사용 / 기존 제출 파일 변경 없음
# ============================================================
import pandas as pd

# 기존 선택을 하면 남는 시간이 5분 이하일 때만 실험
# 대회 환경에서 승객의 최소 최대대기시간이 5분이라는 점을 참고한
# 고정 실험값이며, '5분이면 다음 운행이 불가능하다'는 가정은 아님.
ENDGAME_MIN = 5


def make_endgame_policy():
    base_policy, _ = make_safe_switch_policy()
    decisions = []
    stats = {"검토상태": 0}

    def policy(env, choices):
        base_action = base_policy(env, choices)

        # 후보가 0~1명이면 수락/대기 판단을 바꾸지 않음
        if len(choices) < 2:
            return base_action

        base_r, base_p, base_d = choices[base_action - 1]
        base_finish = env.time + base_p + base_d
        base_left = END - base_finish

        # 기준 정책의 운행이 마감 가까이 끝나는 경우만 검토
        if base_left > ENDGAME_MIN:
            return base_action

        stats["검토상태"] += 1
        base_net = estimate_candidate_net(env, base_r, base_p)

        best_action = base_action
        best_net = base_net

        for action, (r, p, d) in enumerate(choices, start=1):
            net = estimate_candidate_net(env, r, p)

            # 동점이면 기존 정책 유지. 실제 순수익이 클 때만 교체.
            if net > best_net:
                best_net = net
                best_action = action

        if best_action != base_action:
            r, p, d = choices[best_action - 1]
            decisions.append({
                "day": int(env.day),
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "기존순번": base_action,
                "변경순번": best_action,
                "기존하차후_남은분": int(base_left),
                "변경하차후_남은분": int(END - env.time - p - d),
                "즉시순수익차이": int(best_net - base_net),
            })

        return best_action

    return policy, decisions, stats


rows = []
details = []

for day in FIT_DAYS:
    # 비교 기준도 같은 공식 환경에서 직접 재실행
    reference_policy, _ = make_safe_switch_policy()
    ref, _ = run_policy_day(graph, train_requests, day, reference_policy)

    policy, decisions, stats = make_endgame_policy()
    new, _ = run_policy_day(graph, train_requests, day, policy)

    day_gain = int(new["순수익"] - ref["순수익"])
    rows.append({
        "day": day,
        "기준순수익": int(ref["순수익"]),
        "새순수익": int(new["순수익"]),
        "하루순수익차이": day_gain,
        "마감복수후보_검토횟수": stats["검토상태"],
        "교체횟수": len(decisions),
        "승객수차이": int(new["승객수"] - ref["승객수"]),
    })
    details.extend(decisions)

    if not decisions:
        assert day_gain == 0, "선택 변경이 없는데 결과가 다릅니다."

endgame_cmp = pd.DataFrame(rows)
print("[Day 1·2 마감 정책 비교]")
print(endgame_cmp.to_string(index=False))
print(f"\n합산 순수익 차이: {int(endgame_cmp['하루순수익차이'].sum()):+,}원")

if details:
    print("\n[실제 선택을 바꾼 상황]")
    print(pd.DataFrame(details).to_string(index=False))

endgame_pass = (
    endgame_cmp["하루순수익차이"].ge(0).all()
    and endgame_cmp["하루순수익차이"].sum() > 0
)

if endgame_pass:
    print("\n판정: 같은 규칙으로 Day 3 검증을 진행할 가치가 있습니다.")
elif endgame_cmp["교체횟수"].sum() == 0:
    print("\n판정: 두 학습일에서 행동 차이가 없어 우열을 판단할 수 없습니다.")
else:
    print("\n판정: 일관된 개선이 없어 기존 648,249원 정책을 유지합니다.")

exp16_dir = OUTPUT_DIR / "endgame_total_net"
exp16_dir.mkdir(exist_ok=True)
endgame_cmp.to_csv(exp16_dir / "comparison.csv", index=False)
if details:
    pd.DataFrame(details).to_csv(exp16_dir / "decisions.csv", index=False)

[Day 1·2 마감 정책 비교]
 day  기준순수익   새순수익  하루순수익차이  마감복수후보_검토횟수  교체횟수  승객수차이
   1 332727 332727        0            0     0      0
   2 323031 323031        0            0     0      0

합산 순수익 차이: +0원

판정: 두 학습일에서 행동 차이가 없어 우열을 판단할 수 없습니다.


In [34]:
# ============================================================
# 셀 17. 같은 목적지에 더 일찍 도착한 뒤 안전하게 재합류
# 기준 정책: 셀 14의 make_safe_switch_policy()
# ============================================================
import pandas as pd


def can_stay_until(env, pos, early_time, sync_time, known_requests):
    """
    [early_time, sync_time) 동안 action=0이 매번 제자리 대기임을
    현재 알고 있는 정보만으로 보장할 수 있을 때만 True.
    """
    # 이 위치에서 모든 노드가 3-hop 이내라면,
    # 공식 환경의 outside 요청 자체가 존재할 수 없어 자동 이동 불가.
    if max(env.hops[pos].values()) <= 3:
        return True

    # 그 외에는 매분 적어도 하나의 '확실히 유효한 후보'가 필요.
    # 공식 환경은 후보가 하나라도 있으면 action=0에서 자동 이동하지 않음.
    for minute in range(early_time, sync_time):
        has_witness = False

        for r in known_requests:
            if not (r["minute"] <= minute < r["minute"] + r["max_wait"]):
                continue
            if env.hops[pos][r["pickup"]] > 3:
                continue

            _, p = env.shortest_route(pos, r["pickup"], minute)
            pickup_at = minute + p
            if pickup_at >= r["minute"] + r["max_wait"]:
                continue

            _, d = env.shortest_route(r["pickup"], r["dropoff"], pickup_at)
            if pickup_at + d <= END:
                has_witness = True
                break

        if not has_witness:
            return False

    return True


def make_sync_policy():
    base_policy, _ = make_safe_switch_policy()
    sync = None
    switches = []

    stats = {
        "같은목적지_더빠른대안": 0,
        "그중_즉시순수익우위": 0,
        "그중_만료조건충족": 0,
    }

    def policy(env, choices):
        nonlocal sync

        # 교체 후 기존 하차 시각까지 의도적으로 제자리 대기
        if sync is not None:
            target_time, target_pos = sync
            assert env.pos == target_pos, "재합류 전에 예상치 못한 이동 발생"
            assert env.time <= target_time, "재합류 시각을 지나쳤습니다."

            if env.time < target_time:
                return 0

            sync = None  # 원래 하차 위치·시각에 합류

        base_action = base_policy(env, choices)
        if len(choices) < 2:
            return base_action

        r0, p0, d0 = choices[base_action - 1]
        base_end = int(env.time + p0 + d0)
        base_pos = int(r0["dropoff"])
        base_net = estimate_candidate_net(env, r0, p0)
        base_expiry = int(r0["minute"] + r0["max_wait"])

        best_action = base_action
        best_net = base_net
        best_end = None

        for action, (r, p, d) in enumerate(choices, start=1):
            if action == base_action:
                continue

            early_end = int(env.time + p + d)
            if int(r["dropoff"]) != base_pos or early_end >= base_end:
                continue
            stats["같은목적지_더빠른대안"] += 1

            net = estimate_candidate_net(env, r, p)
            if net <= base_net:
                continue
            stats["그중_즉시순수익우위"] += 1

            expiry = int(r["minute"] + r["max_wait"])
            if base_expiry > base_end or expiry > base_end:
                continue
            stats["그중_만료조건충족"] += 1

            if net <= best_net:
                continue

            # 새로 태울 승객은 완료되므로 대기 보장의 근거에서 제외.
            # 아직 발생하지 않은 요청은 읽지 않음.
            known = [
                req for req, _, _ in choices
                if req["request_id"] != r["request_id"]
            ]

            if not can_stay_until(env, base_pos, early_end, base_end, known):
                continue

            best_action = action
            best_net = net
            best_end = early_end

        if best_action != base_action:
            sync = (base_end, base_pos)
            switches.append({
                "day": int(env.day),
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "기존순번": base_action,
                "변경순번": best_action,
                "재합류노드": base_pos,
                "제자리대기분": base_end - best_end,
                "확정이득": int(best_net - base_net),
            })

        return best_action

    return policy, switches, stats


rows, details = [], []

for day in (1, 2, 3):
    base_policy, _ = make_safe_switch_policy()
    ref, _ = run_policy_day(graph, train_requests, day, base_policy)

    policy, switches, stats = make_sync_policy()
    new, _ = run_policy_day(graph, train_requests, day, policy)

    actual_gain = int(new["순수익"] - ref["순수익"])
    accounted_gain = sum(s["확정이득"] for s in switches)

    # 교체로 번 금액과 하루 순수익 차이가 정확히 같아야 함.
    # 다르면 정책을 저장하거나 테스트에 사용하지 않고 즉시 중단.
    assert actual_gain == accounted_gain, (
        f"Day {day}: 실제 차이={actual_gain}, 교체 이득 합={accounted_gain}"
    )

    rows.append({
        "day": day,
        **stats,
        "실제교체": len(switches),
        "순수익차이": actual_gain,
    })
    details.extend(switches)

sync_report = pd.DataFrame(rows)
print("[같은 목적지 조기 도착 → 기존 경로 재합류]")
print(sync_report.to_string(index=False))
print(f"\n합산 확인 이득: {int(sync_report['순수익차이'].sum()):+,}원")

if details:
    print("\n[실제 교체 내역]")
    print(pd.DataFrame(details).to_string(index=False))

sync_dir = OUTPUT_DIR / "safe_early_arrival"
sync_dir.mkdir(exist_ok=True)
sync_report.to_csv(sync_dir / "comparison.csv", index=False)
if details:
    pd.DataFrame(details).to_csv(sync_dir / "switches.csv", index=False)

[같은 목적지 조기 도착 → 기존 경로 재합류]
 day  같은목적지_더빠른대안  그중_즉시순수익우위  그중_만료조건충족  실제교체  순수익차이
   1            0           0          0     0      0
   2            0           0          0     0      0
   3            0           0          0     0      0

합산 확인 이득: +0원


In [40]:
# ============================================================
# 셀 18. 조기 도착·재합류 규칙 최종 확인
# 현재 최고 제출 파일은 덮어쓰지 않음
# ============================================================
best_path = OUTPUT_DIR / "safe_switch_submission" / "resultset.csv"
if not best_path.is_file():
    raise FileNotFoundError(f"648,249원 제출 파일을 찾을 수 없습니다: {best_path}")

best_sub = pd.read_csv(best_path)
assert list(best_sub.columns) == ["id", "day", "step", "passenger_id"]
assert best_sub.shape == (4000, 4)

expected_best = {4: 334_135, 5: 314_114}
candidate_sub = best_sub.copy()
rows, all_switches = [], []


def padded_ids(log):
    ids = log["passenger_id"].to_numpy(dtype=np.int64)
    assert len(ids) <= MAX_STEPS
    assert np.array_equal(log["step"].to_numpy(), np.arange(len(ids)))
    return np.pad(ids, (0, MAX_STEPS - len(ids)), constant_values=-1)


for day in (4, 5):
    idx = best_sub.index[best_sub["day"].eq(day)]
    assert np.array_equal(best_sub.loc[idx, "step"].to_numpy(), np.arange(MAX_STEPS))

    # 1. 비교 기준이 실제 648,249원 제출과 같은 정책인지 확인
    base_policy, _ = make_safe_switch_policy()
    ref, ref_log = run_policy_day(test_graph, test_requests, day, base_policy)

    assert ref["순수익"] == expected_best[day], f"Day {day}: 기준 순수익 불일치"
    assert np.array_equal(
        padded_ids(ref_log),
        best_sub.loc[idx, "passenger_id"].to_numpy(),
    ), f"Day {day}: 기준 행동이 저장된 최고 제출 파일과 다릅니다."

    # 2. 고정된 셀 17 정책을 그대로 운행
    new_policy, switches, stats = make_sync_policy()
    new, new_log = run_policy_day(test_graph, test_requests, day, new_policy)

    actual_gain = int(new["순수익"] - ref["순수익"])
    switch_gain = sum(s["확정이득"] for s in switches)
    assert actual_gain == switch_gain, (
        f"Day {day}: 하루 수익 차이 {actual_gain:+,}원과 "
        f"교체 이득 합 {switch_gain:+,}원이 다릅니다."
    )

    new_ids = padded_ids(new_log)
    if not switches:
        assert np.array_equal(new_ids, padded_ids(ref_log))

    candidate_sub.loc[idx, "passenger_id"] = new_ids
    all_switches.extend(switches)
    rows.append({
        "day": day,
        "기존순수익": ref["순수익"],
        "새순수익": new["순수익"],
        "같은목적지_더빠른대안": stats["같은목적지_더빠른대안"],
        "교체횟수": len(switches),
        "추가이득": actual_gain,
    })

report18 = pd.DataFrame(rows)
gain18 = int(report18["추가이득"].sum())

print(report18.to_string(index=False))
print(f"\n합산 순수익: {int(report18['새순수익'].sum()):,}원")
print(f"648,249원 대비: {gain18:+,}원")

if gain18 > 0:
    print("\n[추가 이득이 발생한 교체]")
    print(pd.DataFrame(all_switches).to_string(index=False))

    new_dir = OUTPUT_DIR / "safe_sync_submission"
    new_dir.mkdir(exist_ok=True)
    new_path = new_dir / "resultset.csv"
    candidate_sub.to_csv(new_path, index=False)
    print("새 제출 파일:", new_path)

    from google.colab import files
    files.download(str(new_path))
else:
    print("\n추가 이득이 없습니다. 새로 제출할 필요가 없습니다.")
    print("유지할 파일:", best_path)

 day  기존순수익   새순수익  같은목적지_더빠른대안  교체횟수  추가이득
   4 334135 334135            0     0     0
   5 314114 314114            0     0     0

합산 순수익: 648,249원
648,249원 대비: +0원

추가 이득이 없습니다. 새로 제출할 필요가 없습니다.
유지할 파일: /content/taxi_run_n4mv6549/output/safe_switch_submission/resultset.csv


In [41]:
# ============================================================
# 셀 19. 공격적 후보 모델: 시간 비용 중심화 + Double DQN
# 전제: 셀 1~18을 실행한 상태
# 기존 최고 제출 파일을 변경하지 않음
# ============================================================
import random
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

seed_all(42)
DEVICE = torch.device("cpu")
torch.set_num_threads(1)

RL_DIR = OUTPUT_DIR / "profit_dqn"
RL_DIR.mkdir(exist_ok=True)

EPISODES = 120
BATCH = 96
WARMUP = 384
CHECK_EVERY = 10
TARGET_EVERY = 150

# 원 단위 → 천 원 단위로 바꾸기 전의 기준 분당 수익
fit_days = (1, 2)
fit_ref = baseline_summary[baseline_summary["day"].isin(fit_days)]
rho = float(fit_ref["순수익"].sum()) / ((END - START) * len(fit_days))
print(f"시간 비용 중심값: {rho:.2f}원/분")


# ------------------------------------------------------------
# 1. Q 모델
# 초기에는 공식 후보 1번을 선호하지만, 학습이 진행되면
# 0~8번 중 어떤 유효 행동이든 선택할 수 있음.
# ------------------------------------------------------------
class ProfitQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(52, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Linear(128, 96),
            nn.SiLU(),
            nn.Linear(96, 9),
        )
        last = self.body[-1]
        nn.init.zeros_(last.weight)
        nn.init.zeros_(last.bias)
        with torch.no_grad():
            last.bias[1] = 0.25  # 초기 동점에 가까운 상황에서 공식 1번 선호

    def forward(self, x):
        return self.body(x)


online = ProfitQ().to(DEVICE)
target = ProfitQ().to(DEVICE)
target.load_state_dict(online.state_dict())
target.eval()

optimizer = torch.optim.AdamW(online.parameters(), lr=3e-4)
replay = deque(maxlen=30000)
updates = 0


@torch.no_grad()
def greedy_action(model, state, mask):
    x = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)[None, :]
    m = torch.as_tensor(mask, dtype=torch.bool, device=DEVICE)
    q = model(x)[0].masked_fill(~m, -torch.inf)
    return int(q.argmax().item())


def exploration_action(model, state, mask, epsilon):
    valid = np.flatnonzero(mask)
    if len(valid) == 1:
        return int(valid[0])

    if random.random() < epsilon:
        # 탐색 시 공식 1번에만 치우치지 않도록 대기와 대체 후보도 경험
        return int(random.choice(valid.tolist()))

    return greedy_action(model, state, mask)


# ------------------------------------------------------------
# 2. 후보가 없어 강제로 대기해야 하는 구간 건너뛰기
# env.step(0)을 실제로 호출하므로 공식 시간/유류비가 그대로 반영됨.
# ------------------------------------------------------------
def advance_forced(env, obs, done):
    while not done and not obs[1][1:].any():
        obs, _, done, _ = env.step(0)
    return obs, done


def optimize_batch():
    global updates

    if len(replay) < WARMUP:
        return

    batch = random.sample(replay, BATCH)
    s, a, r, ns, nm, done = zip(*batch)

    s = torch.as_tensor(np.stack(s), dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(a, dtype=torch.long, device=DEVICE)
    r = torch.as_tensor(r, dtype=torch.float32, device=DEVICE)
    ns = torch.as_tensor(np.stack(ns), dtype=torch.float32, device=DEVICE)
    nm = torch.as_tensor(np.stack(nm), dtype=torch.bool, device=DEVICE)
    done = torch.as_tensor(done, dtype=torch.bool, device=DEVICE)

    q = online(s).gather(1, a[:, None]).squeeze(1)

    with torch.no_grad():
        next_action = online(ns).masked_fill(~nm, -torch.inf).argmax(dim=1)
        next_value = target(ns).gather(1, next_action[:, None]).squeeze(1)

        # 하루 총수익이 목적이므로 미래 순수익을 행동 횟수로 할인하지 않음
        y = r + torch.where(done, torch.zeros_like(next_value), next_value)

    loss = nn.functional.smooth_l1_loss(q, y)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(online.parameters(), 5.0)
    optimizer.step()

    updates += 1
    if updates % TARGET_EVERY == 0:
        target.load_state_dict(online.state_dict())


# ------------------------------------------------------------
# 3. 한 에피소드 학습
# ------------------------------------------------------------
def train_one_day(day, epsilon):
    env = TaxiEnv(graph, train_requests, day)
    obs, done = advance_forced(env, env.observe(), False)

    # 첫 의사결정 전 강제 이동은 모든 정책에 공통.
    # 이후 중심화 보상이 실제 순수익과 일치하는지도 에피소드마다 확인.
    start_earned = env.earned
    first_decision_time = min(env.time, END)
    reward_sum = 0.0

    while not done:
        s, mask = obs
        action = exploration_action(online, s, mask, epsilon)

        t0 = min(env.time, END)
        net0 = env.earned

        obs, _, done, _ = env.step(action)
        obs, done = advance_forced(env, obs, done)

        # 마지막 자동 이동이 22시를 넘겨도 시간 중심화는 영업 종료까지만 계산
        minutes = min(env.time, END) - t0
        delta_net = env.earned - net0
        reward = (delta_net - rho * minutes) / 1000.0
        reward_sum += reward

        ns, next_mask = obs
        replay.append((
            s.copy(), int(action), float(reward),
            ns.copy(), next_mask.copy(), bool(done)
        ))
        optimize_batch()

    expected = (
        env.earned - start_earned - rho * (END - first_decision_time)
    ) / 1000.0
    assert np.isclose(reward_sum, expected), "학습 보상 합계 불일치"


# 평가 때는 강제 구간도 공식 run_policy_day로 실행:
# 실제 대기 횟수와 기존 정책에서 달라진 행동 수 확인
def evaluate_q(model, day):
    @torch.no_grad()
    def policy(env, choices):
        if not choices:
            return 0
        state, mask = env.observe()
        return greedy_action(model, state, mask)

    return run_policy_day(graph, train_requests, day, policy)


# ------------------------------------------------------------
# 4. 학습일에서 체크포인트 선택
# 합산 이득을 우선하되, 한 날짜에서의 손해가 기준의 3%를 넘으면 제외.
# 3%는 학습일 필터일 뿐, 테스트 손실 한도를 보장하지 않음.
# ------------------------------------------------------------
reference_net = {
    int(row["day"]): int(row["순수익"])
    for _, row in fit_ref.iterrows()
}
best_total = -np.inf
best_state = None
best_epoch = None
history = []

for ep in range(1, EPISODES + 1):
    day = fit_days[(ep - 1) % len(fit_days)]
    epsilon = max(0.08, 0.70 * (1.0 - (ep - 1) / (EPISODES * 0.85)))
    train_one_day(day, epsilon)

    if ep % CHECK_EVERY != 0:
        continue

    gains = {}
    total = 0
    for d in fit_days:
        summary, _ = evaluate_q(online, d)
        total += summary["순수익"]
        gains[d] = summary["순수익"] - reference_net[d]

    passes_floor = all(gains[d] >= -0.03 * reference_net[d] for d in fit_days)
    history.append({
        "episode": ep,
        "day1차이": gains[1],
        "day2차이": gains[2],
        "합산차이": sum(gains.values()),
        "손실필터통과": passes_floor,
    })

    if passes_floor and sum(gains.values()) > 0 and total > best_total:
        best_total = total
        best_epoch = ep
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in online.state_dict().items()
        }

    print(
        f"ep={ep:3d} | D1 {gains[1]:+,}원 | D2 {gains[2]:+,}원 "
        f"| 합계 {sum(gains.values()):+,}원 "
        f"| 필터 {'통과' if passes_floor else '미통과'}"
    )

pd.DataFrame(history).to_csv(RL_DIR / "training_history.csv", index=False)


# ------------------------------------------------------------
# 5. 선택된 체크포인트 하나만 Day 3에서 평가
# ------------------------------------------------------------
if best_state is None:
    print("\n학습일에서 합산 개선과 손실 필터를 함께 만족한 모델이 없습니다.")
    print("Day 3과 테스트는 실행하지 않습니다. 기존 648,249원 파일 유지.")
else:
    candidate_q = ProfitQ().to(DEVICE)
    candidate_q.load_state_dict(best_state)
    candidate_q.eval()

    # 기존 최고 정책(셀 14)과 같은 날짜에서 비교
    ref_policy, _ = make_safe_switch_policy()
    ref3, _ = run_policy_day(graph, train_requests, VALID_DAY, ref_policy)
    new3, log3 = evaluate_q(candidate_q, VALID_DAY)

    gain3 = int(new3["순수익"] - ref3["순수익"])
    wait_count = int(log3["kind"].eq("후보거절대기").sum())
    alt_count = int(log3["action"].gt(1).sum())

    print(f"\n선택된 체크포인트: episode {best_epoch}")
    print(f"Day 3 기준 순수익: {ref3['순수익']:,}원")
    print(f"Day 3 DQN 순수익:  {new3['순수익']:,}원")
    print(f"Day 3 차이:        {gain3:+,}원")
    print(f"후보가 있는데 대기: {wait_count}회 | 2번 이상 후보 선택: {alt_count}회")

    torch.save({
        "state_dict": best_state,
        "episode": int(best_epoch),
        "rho": float(rho),
        "day3_gain": gain3,
    }, RL_DIR / "candidate_q.pt")

    if gain3 > 0:
        print("판정: 공격적 후보로 보관. 기존 제출 파일은 변경하지 않았습니다.")
    else:
        print("판정: 학습일 개선이 Day 3에서 재현되지 않았습니다. 기존 제출 유지.")

시간 비용 중심값: 364.31원/분
ep= 10 | D1 -1,321원 | D2 +0원 | 합계 -1,321원 | 필터 통과
ep= 20 | D1 -23,268원 | D2 -33,357원 | 합계 -56,625원 | 필터 미통과
ep= 30 | D1 -14,152원 | D2 -25,670원 | 합계 -39,822원 | 필터 미통과
ep= 40 | D1 -50,410원 | D2 -30,284원 | 합계 -80,694원 | 필터 미통과
ep= 50 | D1 -33,571원 | D2 -27,230원 | 합계 -60,801원 | 필터 미통과
ep= 60 | D1 -24,497원 | D2 -30,362원 | 합계 -54,859원 | 필터 미통과
ep= 70 | D1 -31,809원 | D2 -45,702원 | 합계 -77,511원 | 필터 미통과
ep= 80 | D1 -13,442원 | D2 -24,171원 | 합계 -37,613원 | 필터 미통과
ep= 90 | D1 -13,566원 | D2 -10,784원 | 합계 -24,350원 | 필터 미통과
ep=100 | D1 -26,021원 | D2 -11,668원 | 합계 -37,689원 | 필터 미통과
ep=110 | D1 -25,911원 | D2 -23,530원 | 합계 -49,441원 | 필터 미통과
ep=120 | D1 -24,224원 | D2 -5,029원 | 합계 -29,253원 | 필터 미통과

학습일에서 합산 개선과 손실 필터를 함께 만족한 모델이 없습니다.
Day 3과 테스트는 실행하지 않습니다. 기존 648,249원 파일 유지.


In [43]:
### 셀 20 — 픽업 시간과 승객 운행 시간을 다르게 보는 직접 정책 탐색


# ============================================================
# 셀 20. 하루 순수익 직접 최적화 — 픽업 비용과 건당 시간 조정
# Day 1·2 탐색 → 날짜 교차 확인 → 통과 시에만 Day 3 확인
# 테스트 데이터 및 기존 제출 파일 사용/변경 없음
# ============================================================
from itertools import product
import json
import numpy as np
import pandas as pd

EXP20 = OUTPUT_DIR / "direct_rank_search"
EXP20.mkdir(exist_ok=True)

# 작은 고정 탐색 공간. 실행 결과를 보고 범위를 반복 조정하지 않음.
# a<1도 포함: 픽업을 더 벌주는 방향만 강요하지 않음.
configs20 = list(product(
    (0.5, 1.0, 2.0, 3.0),  # a: 픽업 시간 가중치
    (0.0, 3.0, 8.0),       # h: 건당 고정 시간
    (0.0, 1.0),            # w: 팁·유류비 반영 비율
))
baseline_cfg20 = (1.0, 0.0, 0.0)


def make_direct_rank_policy(cfg):
    a, h, w = cfg
    safe_policy, _ = make_safe_switch_policy()

    def policy(env, choices):
        base_action = safe_policy(env, choices)

        # 후보 없음, 후보 한 명, 또는 공식 1번보다 확실히 유리한 안전 교체
        if len(choices) < 2 or base_action != 1:
            return base_action

        # 기준 설정에서는 점수 재계산 없이 원래 정책과 정확히 일치시킴
        if cfg == baseline_cfg20:
            return base_action

        def score(choice):
            r, pickup, trip = choice
            fare = float(r["base_fare"])
            if w:
                net = float(estimate_candidate_net(env, r, pickup))
                amount = fare + w * (net - fare)
            else:
                amount = fare

            return amount / max(1.0, trip + a * pickup + h)

        return 1 + max(range(len(choices)), key=lambda i: score(choices[i]))

    return policy


# 1. 각 설정을 두 학습일에서 독립적으로 운행
rows20 = []
for j, cfg in enumerate(configs20, start=1):
    row = {"a": cfg[0], "h": cfg[1], "w": cfg[2]}
    for day in FIT_DAYS:
        summary, log = run_policy_day(
            graph, train_requests, day, make_direct_rank_policy(cfg)
        )
        row[f"D{day}"] = int(summary["순수익"])
        row[f"D{day}_1번외선택"] = int(log["action"].gt(1).sum())

    rows20.append(row)
    print(f"\r정책 평가: {j}/{len(configs20)}", end="", flush=True)

print()
tab20 = pd.DataFrame(rows20)

base20 = tab20[
    tab20["a"].eq(1.0) & tab20["h"].eq(0.0) & tab20["w"].eq(0.0)
].iloc[0]

# 셀 14에서 Day 1·2 안전 교체는 0건이었으므로 셀 3 기준과 일치해야 함
for day in FIT_DAYS:
    prev = int(
        baseline_summary.loc[baseline_summary["day"].eq(day), "순수익"].iloc[0]
    )
    assert int(base20[f"D{day}"]) == prev

tab20["D1차이"] = tab20["D1"] - int(base20["D1"])
tab20["D2차이"] = tab20["D2"] - int(base20["D2"])
tab20["합산차이"] = tab20["D1차이"] + tab20["D2차이"]

# 동점이면 기준 정책과 가까운 쪽 선택
tab20["단순성거리"] = (
    np.abs(np.log2(tab20["a"]))
    + tab20["h"] / 8.0
    + tab20["w"]
)

print("\n[학습일 합산 상위 8개 — 이 표만으로 채택하지 않음]")
print(
    tab20.sort_values(
        ["합산차이", "단순성거리"], ascending=[False, True]
    )[["a", "h", "w", "D1차이", "D2차이", "합산차이",
       "D1_1번외선택", "D2_1번외선택"]]
    .head(8).to_string(index=False)
)


# 2. 한 날짜로만 고른 정책을 다른 날짜에서 확인
cross20 = []
for fit_day, check_day in [(1, 2), (2, 1)]:
    chosen = tab20.sort_values(
        [f"D{fit_day}", "단순성거리"], ascending=[False, True]
    ).iloc[0]

    cross20.append({
        "선택에사용한날": fit_day,
        "평가일": check_day,
        "a": chosen["a"],
        "h": chosen["h"],
        "w": chosen["w"],
        "선택일이득": int(chosen[f"D{fit_day}차이"]),
        "다른날실제이득": int(chosen[f"D{check_day}차이"]),
    })

cross20 = pd.DataFrame(cross20)
print("\n[날짜 교차 결과]")
print(cross20.to_string(index=False))

tab20.to_csv(EXP20 / "search.csv", index=False)
cross20.to_csv(EXP20 / "cross_day.csv", index=False)


# 3. 공격적 후보 채택 조건
# 한 날짜에서 최대 3% 손해는 허용하되, 교차평가 합산과 학습 합산은 양수여야 함.
# 이 3%는 다른 날짜나 테스트 손실을 제한한다는 보장이 아님.
cross_ok = (
    cross20["다른날실제이득"].sum() > 0
    and all(
        int(r["다른날실제이득"]) >= -0.03 * int(base20[f"D{int(r['평가일'])}"])
        for _, r in cross20.iterrows()
    )
)

eligible20 = tab20[
    (tab20["합산차이"] > 0)
    & (tab20["D1차이"] >= -0.03 * int(base20["D1"]))
    & (tab20["D2차이"] >= -0.03 * int(base20["D2"]))
]

if not cross_ok or eligible20.empty:
    print("\n판정: 날짜를 바꿨을 때의 성능 또는 학습 기준 미달.")
    print("Day 3과 테스트는 실행하지 않습니다. 648,249원 제출 파일 유지.")
else:
    winner20 = eligible20.sort_values(
        ["합산차이", "단순성거리"], ascending=[False, True]
    ).iloc[0]
    cfg20 = tuple(float(winner20[k]) for k in ("a", "h", "w"))

    ref_policy, _ = make_safe_switch_policy()
    ref3, _ = run_policy_day(graph, train_requests, VALID_DAY, ref_policy)
    cand3, log3 = run_policy_day(
        graph, train_requests, VALID_DAY, make_direct_rank_policy(cfg20)
    )
    gain3 = int(cand3["순수익"] - ref3["순수익"])

    print(f"\n선택 설정: a={cfg20[0]}, h={cfg20[1]}, w={cfg20[2]}")
    print(f"Day 3 기준: {ref3['순수익']:,}원")
    print(f"Day 3 후보: {cand3['순수익']:,}원 | 차이 {gain3:+,}원")
    print(f"Day 3에서 공식 1번 외 선택: {int(log3['action'].gt(1).sum())}회")

    with (EXP20 / "candidate.json").open("w", encoding="utf-8") as f:
        json.dump({
            "a": cfg20[0], "h": cfg20[1], "w": cfg20[2],
            "day3_gain": gain3,
        }, f, ensure_ascii=False, indent=2)

    if gain3 > 0:
        print("판정: 별도 제출 후보로 보관. 기존 최고 제출은 변경하지 않았습니다.")
    else:
        print("판정: Day 3에서 개선되지 않았습니다. 기존 최고 제출 유지.")

정책 평가: 24/24

[학습일 합산 상위 8개 — 이 표만으로 채택하지 않음]
  a   h   w  D1차이  D2차이  합산차이  D1_1번외선택  D2_1번외선택
0.5 0.0 1.0 -4774 10198  5424         3         2
0.5 3.0 1.0 -4781 10198  5417         4         2
1.0 3.0 1.0 -7759 10829  3070         7         6
1.0 0.0 0.0     0     0     0         0         0
0.5 3.0 0.0 -3292  -661 -3953         4         1
2.0 0.0 0.0 -6301  1756 -4545         6         6
1.0 0.0 1.0  -938 -4287 -5225         3         1
1.0 3.0 0.0 -5848     0 -5848         4         0

[날짜 교차 결과]
 선택에사용한날  평가일   a   h   w  선택일이득  다른날실제이득
       1    2 1.0 0.0 0.0      0        0
       2    1 1.0 3.0 1.0  10829    -7759

판정: 날짜를 바꿨을 때의 성능 또는 학습 기준 미달.
Day 3과 테스트는 실행하지 않습니다. 648,249원 제출 파일 유지.


In [44]:
# ============================================================
# 셀 21. 공격적 후보의 별도 평가
# - 셀 20의 보수적 판정은 그대로 유지
# - 후보를 하나로 고정하고 Day 3만 실행
# - 테스트 데이터와 기존 제출 파일은 변경하지 않음
# ============================================================
ATTACK_CFG = (0.5, 0.0, 1.0)

ref_policy, _ = make_safe_switch_policy()
ref21, ref_log21 = run_policy_day(
    graph, train_requests, VALID_DAY, ref_policy
)
attack21, attack_log21 = run_policy_day(
    graph, train_requests, VALID_DAY,
    make_direct_rank_policy(ATTACK_CFG)
)

fields = [
    "순수익", "승객수", "기본요금", "팁", "유류비",
    "픽업이동분", "승객운행분", "자동이동분", "제자리대기분",
]

report21 = pd.DataFrame({
    "기준정책": {k: ref21[k] for k in fields},
    "공격적후보": {k: attack21[k] for k in fields},
})
report21["차이"] = report21["공격적후보"] - report21["기준정책"]

print("[Day 3 비교 | a=0.5, h=0, w=1]")
print(report21.to_string())

gain3 = int(report21.loc["순수익", "차이"])
assert gain3 == (
    report21.loc["기본요금", "차이"]
    + report21.loc["팁", "차이"]
    - report21.loc["유류비", "차이"]
)

# 첫 행동 차이 전까지는 두 정책이 동일한 상태를 방문함.
# 따라서 '정확히 어느 상황에서 경로가 처음 갈라졌는지' 확인 가능.
first_diff = None
for r, a in zip(ref_log21.itertuples(), attack_log21.itertuples()):
    if r.passenger_id != a.passenger_id:
        assert r.start_minute == a.start_minute and r.start_node == a.start_node
        first_diff = (r, a)
        break

if first_diff is None:
    assert len(ref_log21) == len(attack_log21) and gain3 == 0
    print("\nDay 3에서 두 정책의 행동이 전부 같았습니다.")
else:
    r, a = first_diff
    print(
        f"\n첫 선택 차이: {r.start_minute // 60:02d}:{r.start_minute % 60:02d}"
        f" | 노드 {r.start_node} | 후보 {r.candidate_count}명"
    )
    print(f"기준: {r.action}번 후보, 즉시 순수익 {r.net_change:,}원")
    print(f"공격: {a.action}번 후보, 즉시 순수익 {a.net_change:,}원")

print(f"\nDay 3 하루 순수익 차이: {gain3:+,}원")

if gain3 > 0:
    print("판정: 공격적 제출 후보로 보관. 기존 최고 파일은 그대로 유지.")
elif gain3 == 0:
    print("판정: Day 3에서 추가 수익 근거 없음. 기존 최고 정책 우선.")
else:
    print("판정: Day 1과 Day 3 모두 손해. 이 후보는 중단.")

report21.to_csv(EXP20 / "aggressive_day3_comparison.csv")

[Day 3 비교 | a=0.5, h=0, w=1]
          기준정책   공격적후보     차이
순수익     285922  297433  11511
승객수         54      56      2
기본요금    302400  311700   9300
팁        10155   10860    705
유류비      26633   25127  -1506
픽업이동분      209     204     -5
승객운행분      462     474     12
자동이동분      105      98     -7
제자리대기분     124     124      0

첫 선택 차이: 15:14 | 노드 23 | 후보 2명
기준: 1번 후보, 즉시 순수익 7,196원
공격: 2번 후보, 즉시 순수익 5,905원

Day 3 하루 순수익 차이: +11,511원
판정: 공격적 제출 후보로 보관. 기존 최고 파일은 그대로 유지.


In [47]:
# ============================================================
# 셀 22. 공격적 후보 최종 운행
# 고정 설정: a=0.5, h=0, w=1
# 기존 648,249원 제출 파일을 절대 덮어쓰지 않음
# ============================================================
import numpy as np
import pandas as pd
from google.colab import files

seed_all(SEED)
FROZEN_CFG = (0.5, 0.0, 1.0)

# 기존 최고 제출은 형식 확인 및 행동 비교용으로만 읽음
best_path = OUTPUT_DIR / "safe_switch_submission" / "resultset.csv"
best_sub = pd.read_csv(best_path)

assert list(best_sub.columns) == ["id", "day", "step", "passenger_id"]
assert best_sub.shape == (4000, 4)
assert np.array_equal(best_sub["id"].to_numpy(), np.arange(4000))

# 기존 승객 ID를 복사하지 않고 모든 행동을 새 정책으로 생성
attack_sub = best_sub[["id", "day", "step"]].copy()
attack_sub["passenger_id"] = np.full(len(attack_sub), -1, dtype=np.int64)

best_day_net = {4: 334_135, 5: 314_114}
reports = []

for day in (4, 5):
    summary, log = run_policy_day(
        test_graph,
        test_requests,
        day,
        make_direct_rank_policy(FROZEN_CFG),
    )

    idx = attack_sub.index[attack_sub["day"].eq(day)]
    pids = log["passenger_id"].to_numpy(dtype=np.int64)

    assert np.array_equal(attack_sub.loc[idx, "step"].to_numpy(), np.arange(MAX_STEPS))
    assert np.array_equal(log["step"].to_numpy(), np.arange(len(pids)))
    assert 0 < len(pids) <= MAX_STEPS

    served_ids = pids[pids != -1]
    assert len(served_ids) == len(np.unique(served_ids)) == summary["승객수"]

    # 실제 운행만 기록. 종료 뒤의 행은 -1 유지
    attack_sub.loc[idx[:len(pids)], "passenger_id"] = pids

    old_ids = best_sub.loc[idx, "passenger_id"].to_numpy()
    new_ids = attack_sub.loc[idx, "passenger_id"].to_numpy()
    changed = np.flatnonzero(old_ids != new_ids)

    reports.append({
        "day": day,
        "기존최고": best_day_net[day],
        "공격적후보": int(summary["순수익"]),
        "순수익차이": int(summary["순수익"]) - best_day_net[day],
        "승객수": int(summary["승객수"]),
        "유류비": int(summary["유류비"]),
        "첫행동차이_step": int(changed[0]) if len(changed) else None,
    })

report22 = pd.DataFrame(reports)
total = int(report22["공격적후보"].sum())
gain = total - sum(best_day_net.values())

attack_dir = OUTPUT_DIR / "attack_a05_h0_w1"
attack_dir.mkdir(exist_ok=True)
attack_path = attack_dir / "resultset.csv"

attack_sub.to_csv(attack_path, index=False)
report22.to_csv(attack_dir / "test_report.csv", index=False)

print("[고정 정책 Day 4·5 공식 환경 결과]")
print(report22.to_string(index=False))
print(f"\n공격적 후보 합산: {total:,}원")
print(f"기존 648,249원 대비: {gain:+,}원")

if gain > 0:
    print("제출 권장: 공격적 후보가 기존 합산 순수익을 넘었습니다.")
    print("제출할 파일:", attack_path)
    files.download(str(attack_path))
else:
    print("기존 최고 제출을 유지하세요:", best_path)
    print("공격적 후보 파일은 비교용으로만 저장했습니다:", attack_path)

[고정 정책 Day 4·5 공식 환경 결과]
 day   기존최고  공격적후보  순수익차이  승객수   유류비  첫행동차이_step
   4 334135 333753   -382   63 22722          24
   5 314114 292800 -21314   58 26895           7

공격적 후보 합산: 626,553원
기존 648,249원 대비: -21,696원
기존 최고 제출을 유지하세요: /content/taxi_run_n4mv6549/output/safe_switch_submission/resultset.csv
공격적 후보 파일은 비교용으로만 저장했습니다: /content/taxi_run_n4mv6549/output/attack_a05_h0_w1/resultset.csv


In [48]:

# 셀 23. Leave-One-Day-Out 교차검증
# 질문: 이틀에서 선택한 설정이 나머지 하루에서도 수익을 높이는가?
# 필요: 셀 20의 make_direct_rank_policy, 셀 3의 run_policy_day
# ============================================================
from itertools import product
import numpy as np
import pandas as pd

DAYS23 = (1, 2, 3)
BASE23 = (1.0, 0.0, 0.0)
LOSS_LIMIT = 0.03

# 셀 20과 동일한 후보 집합. 테스트 결과를 보고 설정을 추가하지 않음.
GRID23 = list(product(
    (0.5, 1.0, 2.0, 3.0),
    (0.0, 3.0, 8.0),
    (0.0, 1.0),
))
assert len(GRID23) == 24 and BASE23 in GRID23

EXP23 = OUTPUT_DIR / "leave_one_day_out"
EXP23.mkdir(exist_ok=True)

# 각 정책은 고정된 결정 규칙이므로 (설정, 날짜)별 운행은 한 번만 계산.
# 어느 날짜의 점수를 정책 선택에 쓸지는 아래 select_on_two_days에서 제한.
scores23 = {}

def get_score23(cfg, day):
    key = (cfg, day)
    if key not in scores23:
        summary, _ = run_policy_day(
            graph, train_requests, day, make_direct_rank_policy(cfg)
        )
        scores23[key] = int(summary["순수익"])
    return scores23[key]


def distance_from_base23(cfg):
    a, h, w = cfg
    return abs(np.log2(a)) + h / 8.0 + w


def select_on_two_days(fit_days):
    """인자로 받은 두 날짜만 사용. 제외일 점수는 참조하지 않음."""
    assert len(fit_days) == 2

    base = {d: get_score23(BASE23, d) for d in fit_days}
    eligible = []

    for cfg in GRID23:
        gains = [get_score23(cfg, d) - base[d] for d in fit_days]

        # 이틀 중 하루라도 기준 대비 3% 초과 손실이면 제외
        if any(g < -LOSS_LIMIT * base[d] for d, g in zip(fit_days, gains)):
            continue

        eligible.append((cfg, sum(gains)))

    # 기준 정책은 이득 0원으로 항상 포함
    assert any(cfg == BASE23 for cfg, _ in eligible)

    # 1순위: 합산 이득 최대 / 2순위: 기준과 가까운 설정
    # 마지막 cfg는 완전 동점 시 결과를 재현하기 위한 고정 순서
    best_cfg, fit_gain = min(
        eligible,
        key=lambda x: (-x[1], distance_from_base23(x[0]), x[0])
    )
    return best_cfg, int(fit_gain)


rows23 = []

for held_day in DAYS23:
    fit_days = tuple(d for d in DAYS23 if d != held_day)

    # 먼저 나머지 이틀만으로 설정을 확정
    cfg, fit_gain = select_on_two_days(fit_days)

    # 확정한 설정 하나를 제외일에서 기준 정책과 비교
    base_net = get_score23(BASE23, held_day)
    chosen_net = get_score23(cfg, held_day)
    gain = chosen_net - base_net

    rows23.append({
        "평가일": held_day,
        "선택에_쓴_날": ",".join(map(str, fit_days)),
        "a": cfg[0],
        "h": cfg[1],
        "w": cfg[2],
        "선택일_합산이득": fit_gain,
        "평가일_기준": base_net,
        "평가일_선택정책": chosen_net,
        "평가일_차이": gain,
        "평가일_변화율(%)": round(100 * gain / base_net, 2),
    })

cv23 = pd.DataFrame(rows23).sort_values("평가일")
print("[날짜 제외 교차검증]")
print(cv23.to_string(index=False))

total_gain = int(cv23["평가일_차이"].sum())
total_base = int(cv23["평가일_기준"].sum())
worst = cv23.loc[cv23["평가일_차이"].idxmin()]
wins = int(cv23["평가일_차이"].gt(0).sum())
ties = int(cv23["평가일_차이"].eq(0).sum())
losses = int(cv23["평가일_차이"].lt(0).sum())

print("\n[선택에 쓰지 않은 날짜에서의 결과만 합산]")
print(f"합산 이득: {total_gain:+,}원 ({100 * total_gain / total_base:+.2f}%)")
print(
    f"최저 성과일: Day {int(worst['평가일'])} | "
    f"{int(worst['평가일_차이']):+,}원 "
    f"({worst['평가일_변화율(%)']:+.2f}%)"
)
print(f"기준 대비: {wins}승 / {ties}무 / {losses}패")

cv23.to_csv(EXP23 / "held_out_results.csv", index=False)
print("\n저장:", EXP23 / "held_out_results.csv")
print("기존 648,249원 제출 파일은 변경하지 않았습니다.")

[날짜 제외 교차검증]
 평가일 선택에_쓴_날   a   h   w  선택일_합산이득  평가일_기준  평가일_선택정책  평가일_차이  평가일_변화율(%)
   1     2,3 0.5 3.0 1.0     35894  332727    327946   -4781       -1.44
   2     1,3 0.5 3.0 1.0     20915  323031    333229   10198        3.16
   3     1,2 0.5 0.0 1.0      5424  285922    297433   11511        4.03

[선택에 쓰지 않은 날짜에서의 결과만 합산]
합산 이득: +16,928원 (+1.80%)
최저 성과일: Day 1 | -4,781원 (-1.44%)
기준 대비: 2승 / 0무 / 1패

저장: /content/taxi_run_n4mv6549/output/leave_one_day_out/held_out_results.csv
기존 648,249원 제출 파일은 변경하지 않았습니다.


In [49]:
from google.colab import files

files.download(
    '/content/taxi_run_n4mv6549/output/leave_one_day_out/held_out_results.csv'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
# ============================================================
# 셀 24. h=3의 추가 수익이 어디서 나왔는지 경로 분해
# 비교 대상: (a=0.5, h=0, w=1) vs (a=0.5, h=3, w=1)
# Day 1~3만 사용 / 기존 제출 파일 변경 없음
# ============================================================
CFG_H0 = (0.5, 0.0, 1.0)
CFG_H3 = (0.5, 3.0, 1.0)


def compare_horizon_choice(day):
    # A: 하루 내내 h=0
    sum0, log0 = run_policy_day(
        graph, train_requests, day, make_direct_rank_policy(CFG_H0)
    )

    # B: 하루 내내 h=3
    sum3, log3 = run_policy_day(
        graph, train_requests, day, make_direct_rank_policy(CFG_H3)
    )

    first = None
    for old, new in zip(log0.itertuples(), log3.itertuples()):
        if old.passenger_id != new.passenger_id:
            # 이 시점 전까지 같은 행동을 했으므로 같은 상태여야 함
            assert old.start_minute == new.start_minute
            assert old.start_node == new.start_node
            assert old.candidate_count == new.candidate_count
            first = (old, new)
            break

    if first is None:
        assert len(log0) == len(log3)
        assert sum0["순수익"] == sum3["순수익"]
        return {
            "day": day,
            "첫차이시각": "-",
            "첫차이_즉시순수익차": 0,
            "첫선택만변경_하루차이": 0,
            "이후h3유지_추가차이": 0,
            "하루전체_h3차이": 0,
        }

    old, new = first
    flip_step = int(old.step)

    # C: 첫 차이 시점에서만 h=3, 나머지는 모두 h=0
    policy0 = make_direct_rank_policy(CFG_H0)
    policy3 = make_direct_rank_policy(CFG_H3)
    step = 0

    def one_flip_policy(env, choices):
        nonlocal step
        if step == flip_step:
            action = policy3(env, choices)
            assert int(choices[action - 1][0]["request_id"]) == new.passenger_id
        else:
            action = policy0(env, choices)
        step += 1
        return action

    sum_flip, _ = run_policy_day(graph, train_requests, day, one_flip_policy)

    first_effect = int(sum_flip["순수익"] - sum0["순수익"])
    later_effect = int(sum3["순수익"] - sum_flip["순수익"])
    total_effect = int(sum3["순수익"] - sum0["순수익"])
    assert first_effect + later_effect == total_effect

    return {
        "day": day,
        "첫차이시각": f"{old.start_minute // 60:02d}:{old.start_minute % 60:02d}",
        "첫차이_즉시순수익차": int(new.net_change - old.net_change),
        "첫선택만변경_하루차이": first_effect,
        "이후h3유지_추가차이": later_effect,
        "하루전체_h3차이": total_effect,
    }


h_diag = pd.DataFrame([compare_horizon_choice(d) for d in (1, 2, 3)])
print("[h=3에서 h=0 대비 달라진 결과 | 단위: 원]")
print(h_diag.to_string(index=False))

h_diag.to_csv(EXP23 / "h3_vs_h0_path_diagnostic.csv", index=False)

[h=3에서 h=0 대비 달라진 결과 | 단위: 원]
 day 첫차이시각  첫차이_즉시순수익차  첫선택만변경_하루차이  이후h3유지_추가차이  하루전체_h3차이
   1 11:41        1352         -590          583         -7
   2     -           0            0            0          0
   3 07:39        3257        16358        -2173      14185


In [51]:
# ============================================================
# 셀 25. 첫 선택 변경 후 실제 다음 영업까지의 시간 비교
# Day 1·3만 진단 / 미래 정보는 사후 분석에만 사용
# 테스트 및 제출 파일 변경 없음
# ============================================================
def inspect_first_branch(day):
    cfg0 = (0.5, 0.0, 1.0)
    cfg3 = (0.5, 3.0, 1.0)

    _, log0 = run_policy_day(
        graph, train_requests, day, make_direct_rank_policy(cfg0)
    )
    _, log3 = run_policy_day(
        graph, train_requests, day, make_direct_rank_policy(cfg3)
    )

    diff = next(
        (
            (a, b) for a, b in zip(log0.itertuples(), log3.itertuples())
            if a.passenger_id != b.passenger_id
        ),
        None,
    )
    if diff is None:
        return []

    old, new = diff
    flip_step = int(old.step)
    p0 = make_direct_rank_policy(cfg0)
    p3 = make_direct_rank_policy(cfg3)
    counter = 0

    def flip_once(env, choices):
        nonlocal counter
        policy = p3 if counter == flip_step else p0
        counter += 1
        return policy(env, choices)

    _, log_flip = run_policy_day(graph, train_requests, day, flip_once)

    # 선택된 실제 승객의 하차 노드를 운행 후 확인하기 위한 조회표
    drop_lookup = (
        train_requests.loc[train_requests["day"].eq(day)]
        .set_index("request_id")["dropoff"]
    )

    rows = []
    for name, log in [("기존 h=0", log0), ("첫 선택만 h=3", log_flip)]:
        ride = log.loc[log["step"].eq(flip_step)].iloc[0]
        assert ride["action"] > 0

        after_drop = int(ride["start_minute"] + ride["elapsed_min"])
        following = log.loc[log["step"] > flip_step]
        next_rides = following.loc[following["action"] > 0]

        result = {
            "day": day,
            "분기": name,
            "첫승객_하차노드": int(drop_lookup.loc[ride["passenger_id"]]),
            "첫승객_하차시각": f"{after_drop // 60:02d}:{after_drop % 60:02d}",
            "첫승객_순수익": int(ride["net_change"]),
            "하차후_다음수락까지분": None,
            "하차후_다음픽업까지분": None,
            "공백중_자동이동유류비": None,
            "다음승객_순수익": None,
            "첫승객하차후_남은하루순수익": int(following["net_change"].sum()),
        }

        if not next_rides.empty:
            nxt = next_rides.iloc[0]
            gap = int(nxt["start_minute"] - after_drop)
            between = following.loc[following["step"] < nxt["step"]]

            result.update({
                "하차후_다음수락까지분": gap,
                "하차후_다음픽업까지분": gap + int(nxt["pickup_min"]),
                # 수락 전 공백에는 대기 또는 빈차 자동 이동만 있음
                "공백중_자동이동유류비": int(-between["net_change"].sum()),
                "다음승객_순수익": int(nxt["net_change"]),
            })

        rows.append(result)

    return rows


branch25 = pd.DataFrame([
    row
    for day in (1, 3)
    for row in inspect_first_branch(day)
])

print("[첫 선택 이후에는 두 분기 모두 h=0 사용]")
print(branch25.to_string(index=False))
branch25.to_csv(EXP23 / "first_branch_next_service.csv", index=False)

[첫 선택 이후에는 두 분기 모두 h=0 사용]
 day        분기  첫승객_하차노드 첫승객_하차시각  첫승객_순수익  하차후_다음수락까지분  하차후_다음픽업까지분  공백중_자동이동유류비  다음승객_순수익  첫승객하차후_남은하루순수익
   1    기존 h=0        25    11:49     3804            0            4            0      3654          207601
   1 첫 선택만 h=3        19    11:52     5156            0            2            0      4065          205659
   3    기존 h=0         5    07:55     5146            4            4          289      5610          272753
   3 첫 선택만 h=3        10    08:03     8403            8           11          490      4862          285854


In [52]:
# ============================================================
# 셀 26. 45분 반사실 이득으로 학습한 보수적 승객 재선택 정책
# 사용: Day 1~3 / 미사용: Day 4~5, 테스트 결과, 기존 제출 파일
# 필요 함수: run_policy_day, make_safe_switch_policy, pair_features
# ============================================================
HORIZON26 = 45
DAYS26 = (1, 2, 3)
EXP26 = OUTPUT_DIR / "local_horizon_policy"
EXP26.mkdir(exist_ok=True)


def window_net26(log, start_step, end_minute):
    """동일한 벽시계 시각까지의 학습용 순수익 근사치."""
    result = 0.0
    part = log.loc[log["step"] >= start_step]

    for r in part.itertuples():
        t = int(r.start_minute)
        if t >= end_minute:
            break

        length = int(r.elapsed_min)
        fraction = 1.0 if length == 0 else min(1.0, (end_minute - t) / length)
        result += float(r.net_change) * fraction

    return result


def build_day26(day):
    # 기준은 현재 최고 정책. 안전 교체가 적용되는 상황은 학습으로 덮어쓰지 않음.
    safe_ref, _ = make_safe_switch_policy()
    contexts = {}
    k = 0

    def reference(env, choices):
        nonlocal k
        action = safe_ref(env, choices)
        if len(choices) >= 2 and action == 1:
            contexts[k] = (
                int(env.time),
                int(choices[1][0]["request_id"]),
                pair_features(env, choices),
            )
        k += 1
        return action

    ref_summary, ref_log = run_policy_day(graph, train_requests, day, reference)
    X, y = [], []

    for flip_step, (minute, expected_pid, x) in contexts.items():
        safe_after, _ = make_safe_switch_policy()
        counter = 0

        def flip_once(env, choices):
            nonlocal counter
            step = counter
            counter += 1
            if step == flip_step:
                assert len(choices) >= 2
                assert int(choices[1][0]["request_id"]) == expected_pid
                return 2
            return safe_after(env, choices)

        _, alt_log = run_policy_day(graph, train_requests, day, flip_once)
        cutoff = min(END, minute + HORIZON26)
        gain = (
            window_net26(alt_log, flip_step, cutoff)
            - window_net26(ref_log, flip_step, cutoff)
        )
        X.append(x)
        y.append(gain)

    return np.stack(X), np.asarray(y), int(ref_summary["순수익"])


samples26 = {}
for day in DAYS26:
    samples26[day] = build_day26(day)
    _, gains, _ = samples26[day]
    print(
        f"Day {day}: {len(gains)}개 상황 | "
        f"2번의 45분 근사이득 중앙값 {np.median(gains):+,.0f}원"
    )


def fit26(fit_days):
    X = np.concatenate([samples26[d][0] for d in fit_days])
    y = np.concatenate([samples26[d][1] for d in fit_days])

    mu, sd = X.mean(axis=0), X.std(axis=0)
    sd[sd < 1e-6] = 1.0
    Z = (X - mu) / sd

    # 셀 11과 같은 단순 모델·규제·행동 기준 유지
    y = np.clip(y, -10000, 10000)
    bias = float(y.mean())
    weights = np.linalg.solve(
        Z.T @ Z + 20.0 * np.eye(Z.shape[1]),
        Z.T @ (y - bias)
    )
    return mu, sd, weights, bias


def make_policy26(model):
    mu, sd, weights, bias = model
    safe, _ = make_safe_switch_policy()

    def policy(env, choices):
        base = safe(env, choices)
        if len(choices) < 2 or base != 1:
            return base

        # 입력은 현재 후보 정보뿐. 이후 45분 기록은 추론에서 사용하지 않음.
        x = pair_features(env, choices)
        gain = float(((x - mu) / sd) @ weights + bias)
        return 2 if gain > 2000.0 else 1

    return policy


rows26 = []
for held in DAYS26:
    fit_days = tuple(d for d in DAYS26 if d != held)
    model = fit26(fit_days)
    summary, log = run_policy_day(
        graph, train_requests, held, make_policy26(model)
    )
    ref_net = samples26[held][2]
    rows26.append({
        "평가일": held,
        "기준순수익": ref_net,
        "모델순수익": summary["순수익"],
        "공식순수익차이": int(summary["순수익"] - ref_net),
        "2번선택횟수": int(log["action"].eq(2).sum()),
    })

cv26 = pd.DataFrame(rows26)
print("\n[제외일에서의 공식 하루 순수익]")
print(cv26.to_string(index=False))
print(f"\n합산 차이: {int(cv26['공식순수익차이'].sum()):+,}원")
cv26.to_csv(EXP26 / "cross_day.csv", index=False)

Day 1: 31개 상황 | 2번의 45분 근사이득 중앙값 -1,790원
Day 2: 29개 상황 | 2번의 45분 근사이득 중앙값 -1,078원
Day 3: 19개 상황 | 2번의 45분 근사이득 중앙값 -502원

[제외일에서의 공식 하루 순수익]
 평가일  기준순수익  모델순수익  공식순수익차이  2번선택횟수
   1 332727 332727        0       0
   2 323031 323031        0       0
   3 285922 285922        0       0

합산 차이: +0원


In [53]:
# ============================================================
# 셀 27. 승객 순위 예측 대신 하차 후 상태가치 V(시각, 위치) 학습
# 날짜 제외 검증 / 공식 TaxiEnv 불변 / 테스트 미사용
# ============================================================
import numpy as np
import pandas as pd

DAYS27 = (1, 2, 3)
TAU27 = 45.0          # 45분 뒤 수익의 현재 가중치 = exp(-1)
N_ROLLOUT27 = 12      # 날짜당 경로 수. 독립적인 학습일 12개라는 뜻은 아님.
NODES27 = sorted(int(n) for n in graph.nodes)
HOPS27 = dict(nx.all_pairs_shortest_path_length(graph))
EXP27 = OUTPUT_DIR / "post_drop_value"
EXP27.mkdir(exist_ok=True)


# ------------------------------------------------------------
# 1. 공식 환경에서 실제 경로 수집
# 첫 경로는 기준 정책, 나머지는 가끔 다른 승객을 선택.
# 의도적 대기는 추가하지 않아 승객 선택 효과만 실험.
# ------------------------------------------------------------
def collect_value_data(day):
    records = []

    for ep in range(N_ROLLOUT27):
        rng = np.random.default_rng(42000 + 100 * day + ep)
        safe, _ = make_safe_switch_policy()

        def behavior(env, choices):
            a = safe(env, choices)
            if len(choices) >= 2 and ep > 0 and rng.random() < 0.20:
                return int(rng.integers(1, len(choices) + 1))
            return a

        _, log = run_policy_day(graph, train_requests, day, behavior)

        # G(t) = 해당 시점 이후 실제로 번 순수익의 시간 할인 합.
        # 요금/연료비는 행동 완료 시점에 정산하는 학습용 시간 규약.
        # 최종 평가는 이 값이 아니라 공식 env.earned.
        G = 0.0
        for r in reversed(list(log.itertuples(index=False))):
            duration = max(0, int(r.elapsed_min))
            discount = np.exp(-duration / TAU27)
            G = discount * (float(r.net_change) + G)
            records.append((day, int(r.start_minute), int(r.start_node), G))

    df = pd.DataFrame(records, columns=["day", "minute", "node", "target"])

    # 동일한 날짜의 같은 노드·시각을 여러 번 방문해도
    # 독립적인 수요 표본이 여러 개였던 것처럼 중복 계산하지 않음.
    return (
        df.groupby(["day", "minute", "node"], as_index=False)
          .agg(target=("target", "mean"))
    )


value_data27 = {}
for day in DAYS27:
    value_data27[day] = collect_value_data(day)
    print(f"Day {day}: 서로 다른 (시각, 위치) 관측 {len(value_data27[day]):,}개")


# ------------------------------------------------------------
# 2. 제외일을 쓰지 않고 시간 가치와 위치 보정치를 학습
# 공통 시간가치 + 학습일의 실제 잔여 수익으로 추정한 보정치.
# ------------------------------------------------------------
def fit_value27(fit_days):
    df = pd.concat([value_data27[d] for d in fit_days], ignore_index=True)

    # 한 날짜에서 더 많은 위치를 방문했다고 그 날짜가 압도하지 않도록
    # 15분 구간·날짜마다 표본 가중치 합을 1로 맞춤.
    df["slot"] = df["minute"] // 15
    count = df.groupby(["day", "slot"])["node"].transform("size")
    obs_w = 1.0 / count.to_numpy(dtype=float)

    t = df["minute"].to_numpy(dtype=float)
    n = df["node"].to_numpy(dtype=int)
    y = df["target"].to_numpy(dtype=float)

    # 영업 종료에 가까워지면 미래 수익이 0으로 접근하도록 사전값 구성.
    revenue = 0
    for d in fit_days:
        safe, _ = make_safe_switch_policy()
        s, _ = run_policy_day(graph, train_requests, d, safe)
        revenue += s["순수익"]
    rho = revenue / (len(fit_days) * (END - START))

    def prior(minute):
        remaining = np.maximum(0.0, END - minute)
        return rho * TAU27 * (1.0 - np.exp(-remaining / TAU27))

    residual = y - prior(t)

    # 5분 간격 표를 만든 뒤 실제 도착시각에서는 선형 보간
    grid = np.arange(START, END + 1, 5)
    time_values = np.empty(len(grid))
    local_values = {pos: np.empty(len(grid)) for pos in NODES27}

    # 노드 번호의 숫자 차이가 아니라 도로망 hop 거리로 인접성을 표현
    space = {
        pos: np.exp(-np.array([HOPS27[pos][int(j)] for j in n]) / 0.9)
        for pos in NODES27
    }

    for k, minute in enumerate(grid):
        wt = obs_w * np.exp(-np.abs(t - minute) / 40.0)
        global_res = np.dot(wt, residual) / (wt.sum() + 1.0)
        # 종료 시각의 후속 가치는 반드시 0
        fade = min(1.0, max(0.0, (END - minute) / 15.0))
        common = float(prior(minute) + fade * global_res)
        time_values[k] = common

        for pos in NODES27:
            wl = wt * space[pos]
            # 위치 표본이 적으면 시간 전용 모델 쪽으로 수축
            correction = np.dot(wl, residual - global_res) / (wl.sum() + 2.0)
            local_values[pos][k] = common + fade * correction

    def value(minute, pos, use_location):
        if minute >= END:
            return 0.0
        arr = local_values[int(pos)] if use_location else time_values
        return float(np.interp(minute, grid, arr))

    return value


# ------------------------------------------------------------
# 3. 모든 후보에 대해 '현재 운행 + 하차 후 상태가치' 비교
# 안전 교체가 확정된 상태는 그대로 보존.
# ------------------------------------------------------------
def make_value_policy27(value, use_location):
    safe, _ = make_safe_switch_policy()

    def policy(env, choices):
        original = safe(env, choices)
        if len(choices) < 2 or original != 1:
            return original

        scores = []
        for r, p, d in choices:
            duration = p + d
            arrival = env.time + duration
            net = estimate_candidate_net(env, r, p)
            future = value(arrival, r["dropoff"], use_location)
            scores.append(np.exp(-duration / TAU27) * (net + future))

        return int(np.argmax(scores)) + 1

    return policy


# ------------------------------------------------------------
# 4. 다른 이틀로 가치표 학습 → 제외한 하루 실제 운행
# ------------------------------------------------------------
rows27 = []

for held in DAYS27:
    fit_days = tuple(d for d in DAYS27 if d != held)
    value = fit_value27(fit_days)

    safe, _ = make_safe_switch_policy()
    ref, _ = run_policy_day(graph, train_requests, held, safe)

    row = {"평가일": held, "기준순수익": int(ref["순수익"])}

    for name, use_pos in [("시간전용", False), ("시간위치", True)]:
        policy = make_value_policy27(value, use_pos)
        out, log = run_policy_day(graph, train_requests, held, policy)

        row[f"{name}_순수익"] = int(out["순수익"])
        row[f"{name}_차이"] = int(out["순수익"] - ref["순수익"])
        row[f"{name}_1번외선택"] = int(log["action"].gt(1).sum())

    row["위치학습_추가효과"] = row["시간위치_순수익"] - row["시간전용_순수익"]
    rows27.append(row)

cv27 = pd.DataFrame(rows27)
print("\n[학습에 사용하지 않은 날짜의 공식 순수익]")
print(cv27.to_string(index=False))

print("\n[3일 합산]")
for col in ["시간전용_차이", "시간위치_차이", "위치학습_추가효과"]:
    print(f"{col}: {int(cv27[col].sum()):+,}원")

cv27.to_csv(EXP27 / "held_out_results.csv", index=False)

Day 1: 서로 다른 (시각, 위치) 관측 849개
Day 2: 서로 다른 (시각, 위치) 관측 751개
Day 3: 서로 다른 (시각, 위치) 관측 755개

[학습에 사용하지 않은 날짜의 공식 순수익]
 평가일  기준순수익  시간전용_순수익  시간전용_차이  시간전용_1번외선택  시간위치_순수익  시간위치_차이  시간위치_1번외선택  위치학습_추가효과
   1 332727    310163   -22564           5    324409    -8318           5      14246
   2 323031    318744    -4287           1    318744    -4287           2          0
   3 285922    294946     9024           2    295076     9154           3        130

[3일 합산]
시간전용_차이: -17,827원
시간위치_차이: -3,451원
위치학습_추가효과: +14,376원


In [54]:
# ============================================================
# 셀 28. 공식 순위 + 학습된 위치 잔차만 사용
# 전제: 셀 27 실행 완료
# Day 1~3 날짜 제외 평가 / 테스트 미사용 / 제출 파일 변경 없음
# ============================================================
EXP28 = OUTPUT_DIR / "baseline_plus_location_residual"
EXP28.mkdir(exist_ok=True)


def make_residual_policy28(value, use_location=True):
    safe, _ = make_safe_switch_policy()
    changes = []

    def policy(env, choices):
        base_action = safe(env, choices)

        # 후보가 0~1명이거나 이미 확정 이득인 안전 교체가 있으면 그대로 유지
        if len(choices) < 2 or base_action != 1:
            return base_action

        first, p0, d0 = choices[0]
        first_rate = float(first["base_fare"]) / max(1, p0 + d0)

        def location_premium(r, pickup, trip):
            if not use_location:
                return 0.0

            duration = pickup + trip
            arrival = env.time + duration

            # 같은 시각의 일반적 가치 제거 → 위치 때문에 달라지는 몫만 남김
            loc = value(arrival, r["dropoff"], True)
            common = value(arrival, r["dropoff"], False)
            return float(np.exp(-duration / TAU27) * (loc - common))

        premium0 = location_premium(first, p0, d0)
        best_action = 1
        best_advantage = 0.0
        best_details = None

        for action, (r, p, d) in enumerate(choices[1:], start=2):
            # 공식 1번과 같은 분당 기본요금을 내기 위해 부족한 금액
            hurdle = max(
                0.0,
                first_rate * max(1, p + d) - float(r["base_fare"])
            )

            location_gain = location_premium(r, p, d) - premium0
            advantage = location_gain - hurdle

            # 위치가 같거나 차이가 미미하면 무조건 기존 1번 유지
            if advantage > best_advantage + 1e-8:
                best_action = action
                best_advantage = advantage
                best_details = (hurdle, location_gain)

        if best_action != 1:
            hurdle, location_gain = best_details
            changes.append({
                "day": int(env.day),
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "선택순번": best_action,
                "기본요금부족분": round(hurdle),
                "예상위치이득차": round(location_gain),
                "예상순우위": round(best_advantage),
            })

        return best_action

    return policy, changes


rows28, details28 = [], []

for held in DAYS27:
    fit_days = tuple(day for day in DAYS27 if day != held)
    value = fit_value27(fit_days)

    # 1. 실제 기준 정책
    safe, _ = make_safe_switch_policy()
    ref, ref_log = run_policy_day(graph, train_requests, held, safe)

    # 2. 위치를 끄면 수익뿐 아니라 행동 전체가 일치해야 함
    off, _ = make_residual_policy28(value, use_location=False)
    off_result, off_log = run_policy_day(graph, train_requests, held, off)

    assert off_result["순수익"] == ref["순수익"]
    assert np.array_equal(
        off_log["passenger_id"].to_numpy(),
        ref_log["passenger_id"].to_numpy()
    ), f"Day {held}: 위치 보정을 껐는데 기준 행동과 다릅니다."

    # 3. 동일한 정책에서 위치 잔차만 활성화
    on, changes = make_residual_policy28(value, use_location=True)
    result, _ = run_policy_day(graph, train_requests, held, on)

    rows28.append({
        "평가일": held,
        "기준순수익": int(ref["순수익"]),
        "위치잔차정책": int(result["순수익"]),
        "공식순수익차이": int(result["순수익"] - ref["순수익"]),
        "위치때문에교체": len(changes),
        "승객수차이": int(result["승객수"] - ref["승객수"]),
        "유류비차이": int(result["유류비"] - ref["유류비"]),
    })
    details28.extend(changes)

cv28 = pd.DataFrame(rows28)
print("[공식 순위에 위치 잔차만 추가한 결과]")
print(cv28.to_string(index=False))

gain28 = int(cv28["공식순수익차이"].sum())
wins28 = int(cv28["공식순수익차이"].gt(0).sum())
print(f"\n3일 합산 차이: {gain28:+,}원 | 개선 날짜: {wins28}/3")

cv28.to_csv(EXP28 / "held_out_results.csv", index=False)

if details28:
    detail_df28 = pd.DataFrame(details28)
    detail_df28.to_csv(EXP28 / "changes.csv", index=False)
    print("\n[위치 정보로 변경한 선택 — 최대 10개]")
    print(detail_df28.head(10).to_string(index=False))

if gain28 > 0 and wins28 >= 2:
    print("\n판정: 위치 잔차 정책을 추가 검토할 근거가 있습니다.")
else:
    print("\n판정: 위치 잔차만으로도 개선이 재현되지 않아 기존 제출을 유지합니다.")

[공식 순위에 위치 잔차만 추가한 결과]
 평가일  기준순수익  위치잔차정책  공식순수익차이  위치때문에교체  승객수차이  유류비차이
   1 332727  342657     9930        1      1  -1920
   2 323031  323031        0        1      0      0
   3 285922  285922        0        0      0      0

3일 합산 차이: +9,930원 | 개선 날짜: 1/3

[위치 정보로 변경한 선택 — 최대 10개]
 day    시각  선택순번  기본요금부족분  예상위치이득차  예상순우위
   1 14:31     2        0       18     18
   2 20:00     2        0      724    724

판정: 위치 잔차만으로도 개선이 재현되지 않아 기존 제출을 유지합니다.


In [57]:
# ============================================================
# 셀 29. 공식 1위와 정확히 동점인 후보 전수 조사
# Day 1~3만 사용 / 공식 환경 수정 없음 / 테스트 미사용
# ============================================================
EXP29 = OUTPUT_DIR / "exact_tie_audit"
EXP29.mkdir(exist_ok=True)


def audit_ties_day(day):
    fit_days = tuple(d for d in DAYS27 if d != day)
    value = fit_value27(fit_days)

    # 위치 모델은 기준 경로의 해당 상태에서 '무엇을 고를지'만 관측.
    # 실제 기준 운행에는 safe 정책의 행동만 전달한다.
    loc_policy, _ = make_residual_policy28(value, use_location=True)
    safe, _ = make_safe_switch_policy()
    events = []
    step_counter = 0

    def record_ties(env, choices):
        nonlocal step_counter
        step = step_counter
        step_counter += 1

        base_action = safe(env, choices)

        # 기존 +600원을 만든 안전 교체는 이 실험으로 덮어쓰지 않음
        if base_action != 1 or len(choices) < 2:
            return base_action

        def official_rate(choice):
            r, p, d = choice
            return r["base_fare"] / max(1, p + d)

        top_rate = official_rate(choices[0])
        tied = [
            j for j in range(1, len(choices))
            if official_rate(choices[j]) == top_rate
        ]
        if not tied:
            return base_action

        loc_action = loc_policy(env, choices)
        base_req, base_p, base_d = choices[0]
        base_net = estimate_candidate_net(env, base_req, base_p)

        for j in tied:
            r, p, d = choices[j]
            events.append({
                "day": int(day),
                "step": int(step),
                "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
                "대안순번": j + 1,
                "대안ID": int(r["request_id"]),
                "기준하차": int(base_req["dropoff"]),
                "대안하차": int(r["dropoff"]),
                "소요시간차이": int((p + d) - (base_p + base_d)),
                "즉시순수익차이": int(
                    estimate_candidate_net(env, r, p) - base_net
                ),
                "위치모델이_대안선택": bool(loc_action == j + 1),
            })

        return base_action

    ref, _ = run_policy_day(graph, train_requests, day, record_ties)

    # 매 실험은 같은 기준 경로에서 시작.
    # 지정한 한 단계만 대안으로 바꾸고 이후에는 다시 safe 정책.
    for event in events:
        default_policy, _ = make_safe_switch_policy()
        counter = 0

        def flip_once(env, choices):
            nonlocal counter
            step = counter
            counter += 1

            if step == event["step"]:
                action = event["대안순번"]
                assert int(choices[action - 1][0]["request_id"]) == event["대안ID"]
                return action

            return default_policy(env, choices)

        alt, _ = run_policy_day(graph, train_requests, day, flip_once)
        event["한번교체_하루차이"] = int(alt["순수익"] - ref["순수익"])

    return events


events29 = []
for day in DAYS27:
    events29.extend(audit_ties_day(day))

if not events29:
    print("공식 기준 경로에 정확한 1위 동점이 없었습니다.")
    print("셀 28의 '부족분 0원'은 표시 반올림이었을 가능성을 확인하세요.")
else:
    ties29 = pd.DataFrame(events29)
    ties29.to_csv(EXP29 / "all_exact_tie_alternatives.csv", index=False)

    print("[정확한 1위 동점: 대안을 한 번만 선택한 결과]")
    print(ties29[[
        "day", "step", "시각", "대안순번",
        "기준하차", "대안하차", "소요시간차이",
        "즉시순수익차이", "위치모델이_대안선택", "한번교체_하루차이",
    ]].to_string(index=False))

    print("\n[기회 수]")
    print("서로 다른 동점 상태:", len(ties29[["day", "step"]].drop_duplicates()))
    print("비교한 동점 대안:", len(ties29))

    print("\n[위치 모델의 선택 여부별 결과]")
    summary29 = ties29.groupby("위치모델이_대안선택")["한번교체_하루차이"].agg(
        대안수="size",
        이득=lambda x: int((x > 0).sum()),
        손해=lambda x: int((x < 0).sum()),
        동일=lambda x: int((x == 0).sum()),
        중앙값="median",
    )
    print(summary29.to_string())

[정확한 1위 동점: 대안을 한 번만 선택한 결과]
 day  step    시각  대안순번  기준하차  대안하차  소요시간차이  즉시순수익차이  위치모델이_대안선택  한번교체_하루차이
   1    82 14:31     2     1    27      -2    -1206        True       9930
   2    15 08:26     2     6    28       0      390       False      -4287
   2   126 20:00     2    12    17       0        0        True          0

[기회 수]
서로 다른 동점 상태: 3
비교한 동점 대안: 3

[위치 모델의 선택 여부별 결과]
            대안수  이득  손해  동일     중앙값
위치모델이_대안선택                         
False         1   0   1   0 -4287.0
True          2   1   0   1  4965.0


In [58]:
# ============================================================
# 셀 30. 정확한 동점에서 위치 판단이 학습 날짜에 따라 뒤집히는지 검사
# 셀 27의 fit_value27, 셀 29의 ties29 사용
# ============================================================
import numpy as np
import pandas as pd

assert not ties29.empty, "셀 29의 동점 상태 표가 필요합니다."

# fit_value27은 학습일만 사용. 같은 가치표는 한 번만 생성.
value_cache30 = {}

def get_value30(days):
    key = tuple(sorted(days))
    if key not in value_cache30:
        value_cache30[key] = fit_value27(key)
    return value_cache30[key]


def premium30(value, env, choice):
    """셀 28과 동일한 하차 위치 잔차. 단위: 원."""
    r, p, d = choice
    duration = p + d
    arrival = env.time + duration
    loc = value(arrival, r["dropoff"], True)
    common = value(arrival, r["dropoff"], False)
    return float(np.exp(-duration / TAU27) * (loc - common))


rows30 = []

for held in DAYS27:
    day_events = ties29.loc[ties29["day"].eq(held)]
    if day_events.empty:
        continue

    train_a, train_b = [d for d in DAYS27 if d != held]
    va = get_value30((train_a,))
    vb = get_value30((train_b,))
    vab = get_value30((train_a, train_b))

    events_by_step = {
        int(step): part.to_dict("records")
        for step, part in day_events.groupby("step")
    }

    # 진단 중에는 행동을 바꾸지 않아 셀 29와 같은 상태에 도달하게 함
    safe, _ = make_safe_switch_policy()
    counter = [0]

    def inspect_policy(env, choices):
        step = counter[0]
        counter[0] += 1

        for e in events_by_step.get(step, []):
            a = int(e["대안순번"])
            alt = choices[a - 1]
            assert int(alt[0]["request_id"]) == int(e["대안ID"])

            def margin(value):
                return premium30(value, env, alt) - premium30(value, env, choices[0])

            ma, mb, mab = margin(va), margin(vb), margin(vab)
            if ma > 0 and mb > 0:
                verdict = "양쪽 모두 대안"
            elif ma <= 0 and mb <= 0:
                verdict = "양쪽 모두 기존"
            else:
                verdict = "학습일에 따라 반대"

            rows30.append({
                "평가일": held,
                "시각": e["시각"],
                "학습일A": train_a,
                "A_대안우위": round(ma, 1),
                "학습일B": train_b,
                "B_대안우위": round(mb, 1),
                "이틀합친_대안우위": round(mab, 1),
                "판단안정성": verdict,
                "실제_한번교체차이": int(e["한번교체_하루차이"]),
            })

        return safe(env, choices)

    run_policy_day(graph, train_requests, held, inspect_policy)

stability30 = pd.DataFrame(rows30)
assert len(stability30) == len(ties29)
print("[양수: 대안 승객 선호 / 음수: 기존 1번 선호]")
print(stability30.to_string(index=False))
stability30.to_csv(EXP29 / "tie_decision_stability.csv", index=False)

[양수: 대안 승객 선호 / 음수: 기존 1번 선호]
 평가일    시각  학습일A  A_대안우위  학습일B  B_대안우위  이틀합친_대안우위      판단안정성  실제_한번교체차이
   1 14:31     2   107.4     3    10.1       18.1   양쪽 모두 대안       9930
   2 08:26     1    67.7     3  -115.1      -38.9 학습일에 따라 반대      -4287
   2 20:00     1   150.1     3   921.1      724.3   양쪽 모두 대안          0


In [59]:
# ============================================================
# 셀 31. 정확한 공동 1위에서만 만장일치로 교체
# 날짜 제외 전체 운행 검증 → 통과 시 Day 1~3 모델로 후보 고정
# 테스트 데이터 및 기존 제출 파일은 사용/변경하지 않음
# ============================================================
import json
import numpy as np
import pandas as pd

EXP31 = OUTPUT_DIR / "unanimous_exact_tie"
EXP31.mkdir(exist_ok=True)


def make_unanimous_tie_policy(fit_days):
    # 각 날짜의 가치표를 따로 유지해야 '독립된 날짜별 투표'가 됨
    models = [(d, get_value30((d,))) for d in sorted(fit_days)]
    if not models:
        raise ValueError("학습일이 하나 이상 필요합니다.")

    safe, _ = make_safe_switch_policy()
    votes_log = []

    def policy(env, choices):
        base_action = safe(env, choices)
        if len(choices) < 2 or base_action != 1:
            return base_action

        rates = [
            r["base_fare"] / max(1, p + d)
            for r, p, d in choices
        ]
        # 셀 29와 동일: 반올림이나 95% 근접 조건을 사용하지 않음
        tied = [i for i, rate in enumerate(rates) if rate == rates[0]]
        if len(tied) == 1:
            return 1

        votes = []
        for _, value in models:
            # tied는 공식 순서. 모델 점수까지 동점이면 공식 순서 우선.
            winner = max(tied, key=lambda i: premium30(value, env, choices[i]))
            votes.append(winner + 1)

        unanimous = all(v == votes[0] for v in votes)
        action = votes[0] if unanimous else 1

        votes_log.append({
            "day": int(env.day),
            "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
            "동점후보수": len(tied),
            "날짜별투표": " / ".join(
                f"D{d}:{v}번" for (d, _), v in zip(models, votes)
            ),
            "최종행동": action,
            "합의로교체": action != 1,
        })
        return action

    return policy, votes_log


# 1. 제외일을 처음부터 끝까지 실제 운행
rows31, decisions31 = [], []

for held in (1, 2, 3):
    fit_days = tuple(d for d in (1, 2, 3) if d != held)

    ref_policy, _ = make_safe_switch_policy()
    ref, ref_log = run_policy_day(graph, train_requests, held, ref_policy)

    tie_policy, votes = make_unanimous_tie_policy(fit_days)
    new, new_log = run_policy_day(graph, train_requests, held, tie_policy)

    switches = sum(v["합의로교체"] for v in votes)
    if switches == 0:
        assert np.array_equal(
            ref_log["passenger_id"].to_numpy(),
            new_log["passenger_id"].to_numpy(),
        )

    rows31.append({
        "평가일": held,
        "기준순수익": int(ref["순수익"]),
        "합의정책순수익": int(new["순수익"]),
        "하루차이": int(new["순수익"] - ref["순수익"]),
        "방문한동점상태": len(votes),
        "합의로교체": switches,
    })
    decisions31.extend(votes)

cv31 = pd.DataFrame(rows31)
print("[제외일 전체 운행 결과]")
print(cv31.to_string(index=False))
print(f"\n3일 합산 차이: {int(cv31['하루차이'].sum()):+,}원")

if decisions31:
    print("\n[실제 운행 경로에서의 투표]")
    print(pd.DataFrame(decisions31).to_string(index=False))

cv31.to_csv(EXP31 / "full_day_validation.csv", index=False)
pd.DataFrame(decisions31).to_csv(EXP31 / "votes.csv", index=False)

# 2. 세 날 중 손해가 없고 합산 개선이 있으면 '제출 후보'로만 고정
passed31 = bool(cv31["하루차이"].ge(0).all() and cv31["하루차이"].sum() > 0)

if passed31:
    final_tie_policy, final_tie_votes = make_unanimous_tie_policy((1, 2, 3))
    print("\n판정: 3개 날짜 모델의 만장일치 정책을 후보로 고정했습니다.")
else:
    print("\n판정: 전체 운행 조건을 통과하지 못했습니다. 기존 정책 유지.")

(EXP31 / "rule.json").write_text(
    json.dumps({
        "rule": "safe_switch_first_then_unanimous_exact_top_tie",
        "training_days": [1, 2, 3],
        "full_day_check_passed": passed31,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

[제외일 전체 운행 결과]
 평가일  기준순수익  합의정책순수익  하루차이  방문한동점상태  합의로교체
   1 332727   342657  9930        1      1
   2 323031   323031     0        2      1
   3 285922   285922     0        0      0

3일 합산 차이: +9,930원

[실제 운행 경로에서의 투표]
 day    시각  동점후보수         날짜별투표  최종행동  합의로교체
   1 14:31      2 D2:2번 / D3:2번     2   True
   2 08:26      2 D1:2번 / D3:1번     1  False
   2 20:00      2 D1:2번 / D3:2번     2   True

판정: 3개 날짜 모델의 만장일치 정책을 후보로 고정했습니다.


141

In [60]:
# ============================================================
# 셀 32. 3개 날짜 모델의 만장일치 정책 → Day 4·5 최종 평가
# 기존 648,249원 파일은 덮어쓰지 않음
# ============================================================
import numpy as np
import pandas as pd

if not passed31:
    raise RuntimeError("셀 31의 전체 운행 검증을 통과하지 않았습니다.")

seed_all(42)

best_path = OUTPUT_DIR / "safe_switch_submission" / "resultset.csv"
best_sub = pd.read_csv(best_path)
assert list(best_sub.columns) == ["id", "day", "step", "passenger_id"]
assert best_sub.shape == (4000, 4)
assert np.array_equal(best_sub["id"].to_numpy(), np.arange(4000))

best_net = {4: 334_135, 5: 314_114}
candidate_sub = best_sub[["id", "day", "step"]].copy()
candidate_sub["passenger_id"] = np.full(4000, -1, dtype=np.int64)

rows32, votes32 = [], []

for day in (4, 5):
    # 두 날 모두 같은 학습 모델을 사용하고, 운행·투표 로그는 새로 시작
    policy, day_votes = make_unanimous_tie_policy((1, 2, 3))
    summary, log = run_policy_day(test_graph, test_requests, day, policy)

    idx = candidate_sub.index[candidate_sub["day"].eq(day)]
    assert np.array_equal(
        candidate_sub.loc[idx, "step"].to_numpy(), np.arange(MAX_STEPS)
    )

    pids = log["passenger_id"].to_numpy(dtype=np.int64)
    assert len(pids) <= MAX_STEPS
    assert np.array_equal(log["step"].to_numpy(), np.arange(len(pids)))

    served_ids = pids[pids != -1]
    assert len(served_ids) == len(np.unique(served_ids)) == summary["승객수"]

    candidate_sub.loc[idx[:len(pids)], "passenger_id"] = pids

    switches = sum(v["합의로교체"] for v in day_votes)
    gain = int(summary["순수익"]) - best_net[day]

    # 합의 교체가 없으면 기존 최고 정책과 행동·수익이 모두 같아야 함
    if switches == 0:
        assert gain == 0, f"Day {day}: 교체 없이 수익이 달라졌습니다."
        assert np.array_equal(
            candidate_sub.loc[idx, "passenger_id"].to_numpy(),
            best_sub.loc[idx, "passenger_id"].to_numpy(),
        ), f"Day {day}: 교체 없이 제출 행동이 달라졌습니다."

    rows32.append({
        "day": day,
        "기존최고": best_net[day],
        "합의정책": int(summary["순수익"]),
        "차이": gain,
        "동점상태": len(day_votes),
        "합의로교체": switches,
    })
    votes32.extend(day_votes)

report32 = pd.DataFrame(rows32)
total32 = int(report32["합의정책"].sum())
gain32 = total32 - sum(best_net.values())

print("[Day 4·5 공식 환경 결과]")
print(report32.to_string(index=False))
print(f"\n합산 순수익: {total32:,}원 | 기존 대비 {gain32:+,}원")

if votes32:
    print("\n[동점 상태에서의 실제 투표]")
    print(pd.DataFrame(votes32).to_string(index=False))
else:
    print("\n두 날짜 모두 만장일치 규칙을 적용할 동점 상태가 없었습니다.")

report32.to_csv(EXP31 / "test_report.csv", index=False)

if gain32 > 0:
    new_dir = OUTPUT_DIR / "unanimous_tie_submission"
    new_dir.mkdir(exist_ok=True)
    new_path = new_dir / "resultset.csv"
    candidate_sub.to_csv(new_path, index=False)
    print("\n제출 권장 파일:", new_path)

    from google.colab import files
    files.download(str(new_path))
else:
    print("\n기존 최고 제출 유지:", best_path)

[Day 4·5 공식 환경 결과]
 day   기존최고   합의정책  차이  동점상태  합의로교체
   4 334135 334135   0     0      0
   5 314114 314114   0     0      0

합산 순수익: 648,249원 | 기존 대비 +0원

두 날짜 모두 만장일치 규칙을 적용할 동점 상태가 없었습니다.

기존 최고 제출 유지: /content/taxi_run_n4mv6549/output/safe_switch_submission/resultset.csv


In [61]:
# ============================================================
# 셀 33. 전체 후보를 사용하는 경험 기반 Bellman 계획 모델
# 필요: 공식 TaxiEnv, run_policy_day, estimate_candidate_net,
#       make_safe_switch_policy, graph, train_requests, OUTPUT_DIR
# ============================================================
from functools import lru_cache
import numpy as np
import pandas as pd

EXP33 = OUTPUT_DIR / "empirical_bellman"
EXP33.mkdir(exist_ok=True)

DAYS33 = (1, 2, 3)
ROLLS33 = 14
K33 = 24
ITERS33 = 40
TAU33 = 60.0

NODES33 = sorted(int(n) for n in graph.nodes)
NODE_IDX33 = {n: i for i, n in enumerate(NODES33)}
HOPS33 = dict(nx.all_pairs_shortest_path_length(graph))

HOP_MATRIX33 = np.array([
    [HOPS33[a][b] for b in NODES33]
    for a in NODES33
], dtype=np.float64)


# ------------------------------------------------------------
# 1. 현재 보이는 전체 후보를 '즉시 결과가 알려진 행동'으로 변환
# action tuple: (순수익, 걸리는 분, 도착 시각, 도착 노드)
# ------------------------------------------------------------
def visible_options33(env, choices):
    options = []
    for r, p, d in choices:
        dt = int(p + d)
        options.append((
            float(estimate_candidate_net(env, r, p)),
            dt,
            int(env.time + dt),
            int(r["dropoff"]),
        ))
    return tuple(options)


# ------------------------------------------------------------
# 2. 공식 학습 환경에서 여러 경로 수집
# 요청 생성, 환경 전이, 요금 수정 없음.
# 같은 날 같은 위치·시각·행동결과 목록은 중복 저장하지 않음.
# ------------------------------------------------------------
def collect_menus33(day):
    unique = {}

    for ep in range(ROLLS33):
        env = TaxiEnv(graph, train_requests, day)
        rng = np.random.default_rng(330000 + day * 100 + ep)

        # 기준 경로 및 서로 다른 강도의 탐색 경로
        epsilon = (0.0, 0.12, 0.30, 0.55)[ep % 4]

        for _ in range(MAX_STEPS):
            minute = int(env.time)
            pos = int(env.pos)
            choices = env.candidates()

            if choices:
                options = visible_options33(env, choices)
                if rng.random() < epsilon:
                    action = int(rng.integers(1, len(choices) + 1))
                else:
                    action = 1

                _, _, done, info = env.step(action)

                # 선택한 행동에 대해 계산한 결과가 공식 정산과 일치해야 함
                net, dt, at, dst = options[action - 1]
                assert net == info["net"]
                assert at == env.time and dst == env.pos

            else:
                # 선택권 없는 이동도 미래 영업 가치에 중요.
                # 추측하지 않고 실제 공식 환경 결과를 기록.
                before = int(env.earned)
                _, _, done, _ = env.step(0)
                options = ((
                    float(env.earned - before),
                    int(env.time - minute),
                    int(env.time),
                    int(env.pos),
                ),)

            key = (minute, pos, options)
            unique[key] = (minute, pos, options)

            if done:
                break
        else:
            raise RuntimeError(f"Day {day}: MAX_STEPS 초과")

    return list(unique.values())


data33 = {}
for day in DAYS33:
    data33[day] = collect_menus33(day)
    n_menus = len(data33[day])
    n_options = sum(len(row[2]) for row in data33[day])
    print(f"Day {day}: 서로 다른 관측 {n_menus:,}개, 행동 결과 {n_options:,}개")


# ------------------------------------------------------------
# 3. 학습일의 관측만 사용해 미래 상태가치 계산
#
# 각 관측의 타깃:
#   max_a discount(dt) * [현재 운행 순수익 + 도착 상태의 가치]
#
# 후보가 없던 관측은 행동이 1개이므로 자동 이동 손실도 반영됨.
# 가까운 시각·노드의 관측을 모아 '도착 전 예상 가치'로 환산.
# ------------------------------------------------------------
def fit_planner33(fit_days):
    rows = [row for day in fit_days for row in data33[day]]
    minutes = np.array([x[0] for x in rows], dtype=np.float64)
    positions = np.array([NODE_IDX33[x[1]] for x in rows], dtype=np.int64)
    n = len(rows)

    # 기준 분당 수익은 학습일만 사용. 표본이 희박한 곳의 초기 추정치.
    total = 0
    for day in fit_days:
        safe, _ = make_safe_switch_policy()
        summary, _ = run_policy_day(graph, train_requests, day, safe)
        total += summary["순수익"]

    rho = total / ((END - START) * len(fit_days))

    def prior(t):
        remaining = np.maximum(0.0, END - np.asarray(t, dtype=float))
        return rho * TAU33 * (1.0 - np.exp(-remaining / TAU33))

    sample_prior = prior(minutes)

    @lru_cache(maxsize=None)
    def neighbors(t, pos):
        # 같은 노드의 비슷한 시각 우선. 부족하면 인접 노드에서 보완.
        dist = (
            np.abs(minutes - t) / 35.0
            + HOP_MATRIX33[NODE_IDX33[pos], positions] / 1.1
        )
        k = min(K33, n)
        ix = np.argsort(dist, kind="stable")[:k]
        w = np.exp(-dist[ix])
        # 분모의 2는 데이터가 희박할 때 prior 쪽으로 수축시키는 역할
        w = w / (2.0 + w.sum())
        return ix, w

    # 반복 계산에 필요한 실제 후속 상태만 사전 구축
    states = sorted({
        (at, dst)
        for _, _, opts in rows
        for _, _, at, dst in opts
        if at < END
    })
    sid = {s: j for j, s in enumerate(states)}
    terminal = len(states)

    max_actions = max(len(opts) for _, _, opts in rows)
    rewards = np.zeros((n, max_actions), dtype=float)
    discounts = np.zeros((n, max_actions), dtype=float)
    successors = np.full((n, max_actions), terminal, dtype=np.int64)
    valid = np.zeros((n, max_actions), dtype=bool)

    for i, (_, _, opts) in enumerate(rows):
        for j, (net, dt, at, dst) in enumerate(opts):
            valid[i, j] = True
            rewards[i, j] = net
            discounts[i, j] = np.exp(-dt / TAU33)
            if at < END:
                successors[i, j] = sid[(at, dst)]

    ix_table = np.stack([neighbors(t, p)[0] for t, p in states])
    w_table = np.stack([neighbors(t, p)[1] for t, p in states])
    state_prior = prior([t for t, _ in states])
    v = np.append(state_prior, 0.0)  # 마지막 항은 영업 종료 상태

    for _ in range(ITERS33):
        q = discounts * (rewards + v[successors])
        q[~valid] = -np.inf
        best_menu_value = q.max(axis=1)

        residual = best_menu_value - sample_prior
        new_v = state_prior + (w_table * residual[ix_table]).sum(axis=1)
        v = np.append(new_v, 0.0)

    # 마지막 반복 결과로 임의의 관측 가능 위치·시각에서도 예측
    q = discounts * (rewards + v[successors])
    q[~valid] = -np.inf
    final_residual = q.max(axis=1) - sample_prior

    @lru_cache(maxsize=None)
    def value(t, pos):
        if t >= END:
            return 0.0
        ix, w = neighbors(int(t), int(pos))
        return float(prior(t) + np.dot(w, final_residual[ix]))

    return value


# ------------------------------------------------------------
# 4. 실제 정책: 안전 교체 또는 공식 1번을 우선하지 않는다.
# 모든 유효 후보를 새 모델의 점수로 비교한다.
# ------------------------------------------------------------
def make_planner_policy33(value):
    def policy(env, choices):
        if not choices:
            return 0
        if len(choices) == 1:
            return 1

        scores = [
            np.exp(-dt / TAU33) * (net + value(at, dst))
            for net, dt, at, dst in visible_options33(env, choices)
        ]
        return int(np.argmax(scores)) + 1

    return policy


# ------------------------------------------------------------
# 5. 다른 이틀로 학습 → 제외일의 '공식 하루 순수익' 평가
# 학습용 할인이 아니라 실제 env.earned로만 판정.
# ------------------------------------------------------------
rows33 = []

for held in DAYS33:
    fit_days = tuple(d for d in DAYS33 if d != held)
    print(f"\n학습 {fit_days} → 평가 Day {held}", flush=True)

    value = fit_planner33(fit_days)
    policy = make_planner_policy33(value)

    safe, _ = make_safe_switch_policy()
    ref, _ = run_policy_day(graph, train_requests, held, safe)
    new, log = run_policy_day(graph, train_requests, held, policy)

    rows33.append({
        "평가일": held,
        "기준순수익": int(ref["순수익"]),
        "계획모델순수익": int(new["순수익"]),
        "차이": int(new["순수익"] - ref["순수익"]),
        "1번외선택": int(log["action"].gt(1).sum()),
        "승객수차이": int(new["승객수"] - ref["승객수"]),
        "유류비차이": int(new["유류비"] - ref["유류비"]),
    })

cv33 = pd.DataFrame(rows33)
print("\n[제외일 실제 운행 결과]")
print(cv33.to_string(index=False))
print(f"\n3일 합산 차이: {int(cv33['차이'].sum()):+,}원")
print(f"최악의 하루 차이: {int(cv33['차이'].min()):+,}원")
print(f"개선 날짜: {int(cv33['차이'].gt(0).sum())}/3")

cv33.to_csv(EXP33 / "held_out_results.csv", index=False)
print("기존 648,249원 제출 파일은 변경하지 않았습니다.")


Day 1: 서로 다른 관측 1,039개, 행동 결과 1,235개
Day 2: 서로 다른 관측 881개, 행동 결과 1,029개
Day 3: 서로 다른 관측 735개, 행동 결과 801개

학습 (2, 3) → 평가 Day 1

학습 (1, 3) → 평가 Day 2

학습 (1, 2) → 평가 Day 3

[제외일 실제 운행 결과]
 평가일  기준순수익  계획모델순수익     차이  1번외선택  승객수차이  유류비차이
   1 332727   331852   -875      5     -4  -1345
   2 323031   306822 -16209      2     -4    864
   3 285922   297563  11641      2      2  -1636

3일 합산 차이: -5,443원
최악의 하루 차이: -16,209원
개선 날짜: 1/3
기존 648,249원 제출 파일은 변경하지 않았습니다.


In [63]:
# ============================================================
# 셀 34. 실험 A: 만장일치 + 강한 선호도 기반 공격적 승객 재선택
# 필요: 셀 27~31의 가치모델·특성 함수, 공식 환경, 학습 요청
# 테스트 데이터 접근 없음 / 제출 파일 생성 없음
# ============================================================
from pathlib import Path
from functools import lru_cache
import json
import numpy as np
import pandas as pd

seed_all(42)

# ---------- 0. 출력 보호 ----------
EXPECTED_ROOT = Path("/content/taxi_run_n4mv6549/output").resolve()
assert Path(OUTPUT_DIR).resolve() == EXPECTED_ROOT, (
    f"현재 OUTPUT_DIR이 요청 경로와 다릅니다: {OUTPUT_DIR}"
)

A_DIR = EXPECTED_ROOT / "experiment_A_consensus_strength"
A_DIR.mkdir(exist_ok=True)

# 재실행 시 이전 실험 결과도 조용히 덮어쓰지 않음
A_FILES = [
    "threshold_comparison.csv", "decision_log.csv",
    "daily_performance.csv", "experiment_summary.csv",
    "fold_selection.csv", "single_switch_effects.csv",
    "strength_analysis.csv", "protocol.json",
]
occupied = [name for name in A_FILES if (A_DIR / name).exists()]
if occupied:
    raise FileExistsError(
        f"기존 실험 결과가 있습니다: {occupied}\n"
        "결과를 보관하려면 해당 실험 폴더를 다른 이름으로 옮긴 뒤 실행하세요."
    )

A_DAYS = (1, 2, 3)
THRESH = {
    "A0": None,  # 원래 정책: 정확한 공동 1위 안에서 만장일치
    "A1": 0.0,
    "A2": 10.0,
    "A3": 30.0,
    "A4": 50.0,
    "A5": 100.0,
}
train_A = train_requests.loc[train_requests["day"].isin(A_DAYS)].copy()
assert set(train_A["day"].unique()) == set(A_DAYS)

def fresh_run(day, policy):
    """매번 새 graph cache, 새 TaxiEnv, 같은 공식 초기 상태로 시작."""
    if day not in A_DAYS:
        raise ValueError("실험 A에서는 Day 1~3만 허용합니다.")
    g = graph.copy()
    g.graph.pop("route_cache", None)
    return run_policy_day(g, train_A, day, policy)


# ---------- 1. 날짜별 독립 모델 + 공백시간 진단 모델 ----------
@lru_cache(None)
def day_model(d):
    # 셀 27에서 해당 날짜의 공식 학습 경로로 만든 자료만 사용
    assert d in A_DAYS
    return fit_value27((d,))


@lru_cache(None)
def gap_model(d):
    """
    진단용: 하차 후 다음 승객 '선택'까지 걸리는 분을 추정한다.
    다음 승객의 픽업 이동 시간은 포함하지 않는다.
    평가일의 실제 다음 행동을 입력에 쓰지 않는다.
    """
    safe, _ = make_safe_switch_policy()
    _, log = fresh_run(d, safe)
    rides = log.loc[log["action"].gt(0)].reset_index(drop=True)
    lookup = train_A.loc[train_A.day.eq(d)].set_index("request_id")["dropoff"]

    ts, ps, gs = [], [], []
    for j in range(len(rides) - 1):
        r, nxt = rides.iloc[j], rides.iloc[j + 1]
        end = int(r.start_minute + r.elapsed_min)
        ts.append(end)
        ps.append(int(lookup.loc[int(r.passenger_id)]))
        gs.append(float(nxt.start_minute - end))

    if not gs:
        raise ValueError(f"Day {d}: 공백시간 추정 표본이 없습니다.")

    ts = np.asarray(ts)
    gs = np.asarray(gs)
    prior = float(gs.mean())

    @lru_cache(None)
    def predict(t, pos):
        h = np.array([HOPS27[pos][p] for p in ps], dtype=float)
        w = np.exp(-np.abs(ts - t) / 75.0 - h)
        return float((np.dot(w, gs) + 4 * prior) / (w.sum() + 4))

    return predict


# 각 날짜의 기준은 648,249원을 만든 정책을 해당 학습일에서 실행한 값.
# 648,249원 자체를 학습일 점수나 학습 목표에 넣지 않음.
base_A = {}
for d in A_DAYS:
    safe, _ = make_safe_switch_policy()
    base_A[d] = int(fresh_run(d, safe)[0]["순수익"])


# ---------- 2. A0~A5 공통 실행기 ----------
def run_A(day, fit_days, label, detailed=False, reference_path=False):
    """
    reference_path=True: 모델 점수는 계산하되 실제로는 안전 기준 정책만 실행.
    단일 교체의 실제 효과를 같은 출발 상태에서 비교할 때 사용.
    """
    da, db = tuple(fit_days)
    va, vb = day_model(da), day_model(db)
    safe, _ = make_safe_switch_policy()
    threshold = THRESH[label]
    logs = []
    switches = 0
    step = 0

    def policy(env, choices):
        nonlocal switches, step
        k = step
        step += 1
        base = safe(env, choices)
        action = base

        row = {
            "평가일": day, "정책": label, "step": k,
            "minute": int(env.time),
            "시각": f"{env.time // 60:02d}:{env.time % 60:02d}",
            "현재노드": int(env.pos), "후보수": len(choices),
            "모델A학습일": da, "모델B학습일": db,
            "임계값": threshold, "기존행동": base,
        }

        if len(choices) < 2:
            reason = "후보 2명 미만"
        elif base != 1:
            reason = "기존 안전 교체 우선"
        else:
            c1, c2 = choices[0], choices[1]
            r1, p1, d1 = c1
            r2, p2, d2 = c2

            def margin(model):
                return premium30(model, env, c2) - premium30(model, env, c1)

            a, b = float(margin(va)), float(margin(vb))
            row.update({"A점수": a, "B점수": b, "최소선호강도": min(a, b)})

            if label == "A0":
                # 셀 31 그대로: 정확한 공동 1위 전체에 대해 각각 투표
                rates = [
                    r["base_fare"] / max(1, p + trip)
                    for r, p, trip in choices
                ]
                tied = [i for i, rate in enumerate(rates) if rate == rates[0]]
                votes = [
                    max(tied, key=lambda i: premium30(v, env, choices[i]))
                    for v in (va, vb)
                ]
                action = votes[0] + 1 if votes[0] == votes[1] else 1
                reason = "기존 공동1위 만장일치"
            else:
                # 경계는 반드시 strict >. 한쪽만 강하거나 부호가 다르면 유지.
                action = 2 if a > threshold and b > threshold else 1
                reason = "2번 강도조건 통과" if action == 2 else "1번 유지"

            if detailed:
                n1 = estimate_candidate_net(env, r1, p1)
                n2 = estimate_candidate_net(env, r2, p2)
                at1, at2 = int(env.time + p1 + d1), int(env.time + p2 + d2)

                def gap_diff(model_day):
                    gm = gap_model(model_day)
                    return gm(at2, int(r2["dropoff"])) - gm(at1, int(r1["dropoff"]))

                row.update({
                    "1번ID": int(r1["request_id"]), "2번ID": int(r2["request_id"]),
                    "1번픽업": int(r1["pickup"]), "2번픽업": int(r2["pickup"]),
                    "1번하차": int(r1["dropoff"]), "2번하차": int(r2["dropoff"]),
                    "1번픽업분": p1, "2번픽업분": p2,
                    "1번운행분": d1, "2번운행분": d2,
                    "1번기본요금": r1["base_fare"], "2번기본요금": r2["base_fare"],
                    "1번예상순수익": n1, "2번예상순수익": n2,
                    "예상순수익차": n2 - n1,
                    "예상이동시간차": (p2 + d2) - (p1 + d1),
                    "예상공백시간차": (gap_diff(da) + gap_diff(db)) / 2,
                    "pair_features": json.dumps(
                        np.asarray(pair_features(env, choices), dtype=float).tolist()
                    ),
                })

        if reference_path:
            action = base

        switched = action != base
        switches += int(switched)
        if detailed:
            row.update({"변경행동": action, "교체여부": switched, "사유": reason})
            logs.append(row)
        return action

    summary, _ = fresh_run(day, policy)
    return int(summary["순수익"]), switches, logs


# ---------- 3. 임계값 선택은 반드시 제외일 평가 전에 ----------
# 두 학습일로 만든 두 독립 모델을 그 학습일들에서 평가해 임계값 선택.
# 이것은 '학습 내 선택'이므로 낙관적일 수 있음.
# 별도의 외부 성과는 오직 아래 held 날짜로 측정.
def ordered_labels(records, stable=False):
    if stable:
        # 안정형: 최악 날짜 → 합계 → 교체 적음 → 동일하면 A0 우선
        key = lambda x: (-x["worst"], -x["total"], x["count"], x["label"])
    else:
        # 수익형: 합계 → 최악 날짜 → 교체 적음 → 동일하면 A0 우선
        key = lambda x: (-x["total"], -x["worst"], x["count"], x["label"])
    return sorted(records, key=key)


daily, decision_rows, folds = [], [], []

for held in A_DAYS:
    fit = tuple(d for d in A_DAYS if d != held)
    fit_stats = []

    for label in THRESH:
        gains, count = [], 0
        for d in fit:
            net, n, _ = run_A(d, fit, label)
            gains.append(net - base_A[d])
            count += n
        fit_stats.append({
            "label": label, "total": sum(gains),
            "worst": min(gains), "count": count,
        })

    profit_pick = ordered_labels(fit_stats)[0]["label"]
    stable_pick = ordered_labels(fit_stats, stable=True)[0]["label"]

    # 여기부터 제외일 평가. 선택된 임계값은 변경하지 않음.
    fold_result = {}
    for label in THRESH:
        net, count, logs = run_A(held, fit, label, detailed=True)
        fold_result[label] = net
        decision_rows.extend(logs)
        daily.append({
            "날짜": held, "학습일": f"{fit[0]},{fit[1]}",
            "정책": label, "임계값": THRESH[label],
            "기준순수익": base_A[held], "실험순수익": net,
            "기준대비차이": net - base_A[held],
            "실제추가교체횟수": count,
        })

    folds.append({
        "평가일": held, "선택에사용한날": f"{fit[0]},{fit[1]}",
        "수익형_사전선택": profit_pick,
        "수익형_제외일차이": fold_result[profit_pick] - base_A[held],
        "수익형_A0대비": fold_result[profit_pick] - fold_result["A0"],
        "안정형_사전선택": stable_pick,
        "안정형_제외일차이": fold_result[stable_pick] - base_A[held],
        "안정형_A0대비": fold_result[stable_pick] - fold_result["A0"],
    })
    print(f"Day {held} 완료 | 학습일 {fit} | 사전선택: 수익형 {profit_pick}, 안정형 {stable_pick}")


# ---------- 4. 필수 비교표 ----------
daily = pd.DataFrame(daily)
decisions = pd.DataFrame(decision_rows)
folds = pd.DataFrame(folds)
a0_day = daily.loc[daily["정책"].eq("A0")].set_index("날짜")["실험순수익"]
daily["A0대비차이"] = daily["실험순수익"] - daily["날짜"].map(a0_day)

# 이전 셀 31과 A0가 실제로 재현되는지 확인
if "cv31" in globals():
    expected = cv31.set_index("평가일")["합의정책순수익"].sort_index()
    assert a0_day.sort_index().to_dict() == expected.to_dict(), "A0 재현 실패"

comparison = []
for label, part in daily.groupby("정책", sort=True):
    part = part.sort_values("날짜")
    g = part["기준대비차이"]
    v = part["A0대비차이"]
    row = {
        "정책": label, "임계값": THRESH[label],
        "3일합산순수익": int(part["실험순수익"].sum()),
        "기준대비합산차이": int(g.sum()),
        "A0대비개선액": int(v.sum()),
        "최악날짜_기준대비": int(g.min()),
        "최악날짜_A0대비": int(v.min()),
        "기준대비_승": int(g.gt(0).sum()),
        "기준대비_무": int(g.eq(0).sum()),
        "기준대비_패": int(g.lt(0).sum()),
        "A0대비_개선날짜수": int(v.gt(0).sum()),
        "실제추가교체횟수": int(part["실제추가교체횟수"].sum()),
    }
    for _, r in part.iterrows():
        d = int(r["날짜"])
        row[f"Day{d}_순수익"] = int(r["실험순수익"])
        row[f"Day{d}_기준대비차이"] = int(r["기준대비차이"])
    comparison.append(row)

comparison = pd.DataFrame(comparison)
a0_worst = int(comparison.loc[
    comparison["정책"].eq("A0"), "최악날짜_기준대비"
].iloc[0])

def assess(r):
    if r["정책"] == "A0":
        return "비교 기준"
    if r["A0대비개선액"] < 0:
        return "기각: A0보다 낮음"
    if r["A0대비개선액"] == 0:
        return "개선 확인 안 됨: A0와 동점"
    if r["최악날짜_기준대비"] < a0_worst:
        return "보류: 합계 개선, 최악 날짜 악화"
    if r["A0대비_개선날짜수"] == 1:
        return "개선 후보 보존: 한 날짜에 집중, 과적합 주의"
    return "개선 후보 보존: 테스트 성능 미확인"

comparison["판정"] = comparison.apply(assess, axis=1)

profit_best = comparison.sort_values(
    ["3일합산순수익", "최악날짜_기준대비", "실제추가교체횟수", "정책"],
    ascending=[False, False, True, True]
).iloc[0]

stable_best = comparison.sort_values(
    ["최악날짜_기준대비", "3일합산순수익", "실제추가교체횟수", "정책"],
    ascending=[False, False, True, True]
).iloc[0]

summary = pd.DataFrame([
    {"선정기준": "고정정책 사후비교: 최고수익", **profit_best.to_dict()},
    {"선정기준": "고정정책 사후비교: 최고안정", **stable_best.to_dict()},
])


# ---------- 5. 선호 강도와 실제 한 번 교체 효과 ----------
# 정책별 경로가 다르므로 하루 수익을 각 행동의 이득으로 배분하지 않는다.
# 모든 임계값에 공통인 '안전 기준 경로'에서 1→2를 한 번만 바꿔 평가.
effects = []

for held in A_DAYS:
    fit = tuple(d for d in A_DAYS if d != held)
    _, _, ref_logs = run_A(held, fit, "A1", detailed=True, reference_path=True)
    eligible = [r for r in ref_logs if "A점수" in r]

    for event in eligible:
        safe, _ = make_safe_switch_policy()
        step_box = [0]

        def flip_once(env, choices):
            k = step_box[0]
            step_box[0] += 1
            if k == event["step"]:
                assert int(choices[1][0]["request_id"]) == event["2번ID"]
                return 2
            return safe(env, choices)

        net = int(fresh_run(held, flip_once)[0]["순수익"])
        effects.append({
            "평가일": held, "step": event["step"], "시각": event["시각"],
            "A점수": event["A점수"], "B점수": event["B점수"],
            "최소선호강도": event["최소선호강도"],
            "의견충돌": (event["A점수"] > 0) != (event["B점수"] > 0),
            "한번교체_실제하루차이": net - base_A[held],
        })

effects = pd.DataFrame(effects)
effects["강도구간"] = pd.cut(
    effects["최소선호강도"],
    bins=[-np.inf, 0, 10, 30, 50, 100, np.inf],
    labels=["≤0", "(0,10]", "(10,30]", "(30,50]", "(50,100]", ">100"],
)
strength = (
    effects.groupby("강도구간", observed=False)["한번교체_실제하루차이"]
    .agg(
        사례수="size", 평균="mean", 중앙값="median",
        이득건수=lambda s: int(s.gt(0).sum()),
        손해건수=lambda s: int(s.lt(0).sum()),
    )
    .reset_index()
)

# 표본은 같은 날짜와 경로를 공유하므로 상관계수는 기술통계로만 사용
rho_s = effects["최소선호강도"].rank().corr(
    effects["한번교체_실제하루차이"].rank()
)

# ---------- 6. 저장 ----------
for name, df in {
    "threshold_comparison.csv": comparison,
    "decision_log.csv": decisions,
    "daily_performance.csv": daily,
    "experiment_summary.csv": summary,
    "fold_selection.csv": folds,
    "single_switch_effects.csv": effects,
    "strength_analysis.csv": strength,
}.items():
    df.to_csv(A_DIR / name, index=False, encoding="utf-8-sig")

(A_DIR / "protocol.json").write_text(json.dumps({
    "training_days": list(A_DAYS),
    "thresholds": THRESH,
    "test_used": False,
    "score": "셀 30 위치 프리미엄(공식 2번 - 공식 1번), 원",
    "A0": "셀 31과 동일한 정확한 공동1위 만장일치",
    "A1_A5": "안전교체 우선, 동점 제한 없이 공식 2번만 검토",
    "gap_feature": "학습일로 추정한 하차 후 다음 수락까지의 공백 차이, 진단용",
    "selection": "외부 제외일 평가 전, 두 학습일의 성적으로 수익형/안정형 선택",
    "production_submission_created": False,
}, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------- 7. 한국어 결과 요약 ----------
print("\n===== 실험 A: 고정 임계값별 전체 하루 결과 =====")
print(comparison[[
    "정책", "임계값", "3일합산순수익", "기준대비합산차이",
    "A0대비개선액", "최악날짜_기준대비", "실제추가교체횟수",
    "기준대비_승", "기준대비_무", "기준대비_패", "판정"
]].to_string(index=False))

print("\n===== 평가일을 보지 않고 선택한 결과 =====")
print(folds.to_string(index=False))
print(f"수익형 선택 절차의 A0 대비 합산: {folds['수익형_A0대비'].sum():+,}원")
print(f"안정형 선택 절차의 A0 대비 합산: {folds['안정형_A0대비'].sum():+,}원")

print("\n===== 요청한 세 상황: 안전 기준 경로에서 비교 =====")
focus = effects[
    ((effects["평가일"] == 1) & (effects["시각"] == "14:31"))
    | ((effects["평가일"] == 2) & effects["시각"].isin(["08:26", "20:00"]))
]
print(focus.drop(columns="강도구간").round(2).to_string(index=False))

print("\n===== 선호 강도와 한 번 교체 결과 =====")
print(strength.round(1).to_string(index=False))
print(f"최소 선호강도와 실제 이득의 순위상관: {rho_s:.3f} (기술통계)")

print("\n===== 최종 요약 =====")
for title, result in [("최고 수익", profit_best), ("최고 안정", stable_best)]:
    print(
        f"{title}: {result['정책']} | "
        f"3일 순수익 {int(result['3일합산순수익']):,}원 | "
        f"A0 대비 {int(result['A0대비개선액']):+,}원 | "
        f"최악 날짜 기준 대비 {int(result['최악날짜_기준대비']):+,}원 | "
        f"교체 {int(result['실제추가교체횟수'])}회 | {result['판정']}"
    )

print("결과 저장:", A_DIR)
print("Day 4·5 미사용. 648,249원 제출 파일은 수정하거나 교체하지 않았습니다.")

Day 1 완료 | 학습일 (2, 3) | 사전선택: 수익형 A4, 안정형 A0
Day 2 완료 | 학습일 (1, 3) | 사전선택: 수익형 A3, 안정형 A3
Day 3 완료 | 학습일 (1, 2) | 사전선택: 수익형 A0, 안정형 A0


AssertionError: A0 재현 실패

In [64]:

# A0 재현 실패 원인 진단
print("=== 기존 cv31 ===")
print(cv31[["평가일", "합의정책순수익"]].sort_values("평가일"))

print("\n=== 셀 34 기준 정책 ===")
print(pd.DataFrame([
    {"평가일": d, "기준순수익": base_A[d]}
    for d in A_DAYS
]))

print("\n=== 셀 34 A0 결과 ===")
print(
    daily[daily["정책"] == "A0"][
        ["날짜", "기준순수익", "실험순수익", "실제추가교체횟수"]
    ].sort_values("날짜")
)

print("\n=== 차이 ===")
old = cv31.set_index("평가일")["합의정책순수익"].sort_index()
new = daily[daily["정책"] == "A0"].set_index("날짜")["실험순수익"].sort_index()

print(pd.DataFrame({
    "기존_A0": old,
    "신규_A0": new,
    "차이": new - old
}))


=== 기존 cv31 ===
   평가일  합의정책순수익
0    1   342657
1    2   323031
2    3   285922

=== 셀 34 기준 정책 ===
   평가일   기준순수익
0    1  331114
1    2  323064
2    3  305650

=== 셀 34 A0 결과 ===
    날짜   기준순수익   실험순수익  실제추가교체횟수
0    1  331114  342657         1
6    2  323064  323064         1
12   3  305650  305650         0

=== 차이 ===
    기존_A0   신규_A0     차이
1  342657  342657      0
2  323031  323064     33
3  285922  305650  19728


In [65]:

# ============================================================
# 실험 A 제출 후보 추출
# 기존 제출 파일 절대 덮어쓰기 금지
# Day 4~5 공식 환경 검증 후 별도 저장
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/taxi_run_n4mv6549/output")
A_DIR = ROOT / "experiment_A_consensus_strength"

OLD_FILE = ROOT / "safe_switch_submission/resultset.csv"
NEW_DIR = ROOT / "experiment_A_submission_candidate"
NEW_DIR.mkdir(exist_ok=True)

# 1. 기존 제출 파일 확인
assert OLD_FILE.exists(), "기존 최고 제출 파일이 없습니다."

# 2. A0 재현 검증
old_cv = cv31.set_index("평가일")["합의정책순수익"].sort_index()
new_cv = daily[daily["정책"] == "A0"].set_index("날짜")["실험순수익"].sort_index()

if not old_cv.equals(new_cv):
    raise RuntimeError(
        "A0 재현 실패: 정책 구현을 수정하기 전에는 제출 후보를 만들 수 없습니다."
    )

# 3. 실험 결과에서 최고 정책 선택
comparison = pd.read_csv(A_DIR / "threshold_comparison.csv")

best = comparison.sort_values(
    ["3일합산순수익", "최악날짜_기준대비", "실제추가교체횟수"],
    ascending=[False, False, True]
).iloc[0]

BEST_LABEL = str(best["정책"])

print("선택 정책:", BEST_LABEL)
print("3일 합산 순수익:", int(best["3일합산순수익"]))
print("기준 대비:", int(best["기준대비합산차이"]))

# 4. 선택 정책 함수 구성
# 아래 make_policy_A는 셀 34의 run_A 정책 로직을
# 운영용으로 분리한 함수여야 합니다.
# Day 4~5의 학습 정보는 사용하지 않습니다.

raise NotImplementedError(
    "A0 재현 검증 및 운영용 정책 함수 연결 후 실행하세요."
)


RuntimeError: A0 재현 실패: 정책 구현을 수정하기 전에는 제출 후보를 만들 수 없습니다.